# Digital Cash Management Forecast: set up the project

This notebook sets up the Digital Cash Management Forecast in this Dataiku project (it also gives the project
that name), the way it runs in production. The daily flows and the forecast are Parquet files in folders (for speed),
the forecast is then loaded into a SQL table, `final_forecast` (`OUTPUT_SQL_CONNECTION`); the logs and small tables
are CSV files.
It reads 2 warehouse tables, read-only, and creates or writes nothing in the warehouse: the clients
(`EDW.DIM_CUSTOMER`) and the daily postings (`EDW.FCT_STATEMENT_ENTRY`), which the staging recipe sums per client,
day and category in Dataiku. The first run reads each client's whole history; after that, each run reads only the
new days (through the database connection, a `SELECT` with a date filter) and adds them to `daily_flows`. It always
reads them through SQL, through the read-only database connection `EDW_CONNECTION`: at the bank, the warehouse;
on the test machine, a local test database with fake data (`warehouse_simulator.ipynb`). The same code, only the
connection differs. Names, codes and the categories (a mock list until the bank's transaction types are known):
`python/warehouse.py` in the project library.

It forecasts, per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 30, 60 and 90 days after the cut-off day (yesterday: days 1-30, 31-60, 61-90), as
P10, EV and P90, from each client's daily flows of the last 365 days. It builds 4 flow zones, each with an owner:

- **Raw data** and **Staging** (shared): the 2 warehouse tables, read-only, then `clients` (the business clients)
  and `daily_flows` (money in and out per client, day and category): what every model gets. `staging_log` records
  each read.
- **Training** (data scientist): each training run is one model with its own **model ID**, trained with the
  **forecaster** the project variable `forecaster` names, tested on the last 6 periods of 30 days, tracked in Dataiku's
  **Experiment Tracking** and deployed as a new **version** of the saved model `cash_forecast`. A champion /
  challenger test activates the new version if it passes the **promotion rules** (project variables `promotion_*`).
- **Inference** (inference owner): the production model's forecast for every active client, one row per client
  (every row says which model and which load produced it), Polars from start to end: the groups of clients of
  `daily_flows` run side by side (project variable `inference_workers`), each read (its last months only),
  prepared, forecast and written on its own into the folder `forecast` (one subfolder per night, kept). Then it is
  loaded into the SQL table `final_forecast`. The inference log has one row per run, with timings and memory.
The data scientist tries new forecasters outside the flow, in the notebook `forecast_lab.ipynb` (it never touches
production), then sets the project variable `forecaster` and runs **Retrain, test and promote**.

The load test is a project of its own (`build_load_test_project.ipynb`, the same code): nothing of it is here.

The zones only meet through three interfaces: **1. the data** (`daily_flows` and `clients`: staging → training, lab
and inference), **2. the forecaster** (`models.py`: lab → training), **3. the active version of the saved model**
(training → inference). Until the data scientist's models are ready, the forecaster is a placeholder,
`daily_catboost`: ONE CatBoost model that gives P10, EV and P90 together, for every category and the 3 windows. Its
inputs: the client's net cash flow of each of the last 365 days (one column per day), the category and the window (1,
2, 3), the rows the nightly forecast builds (`cashflow.model_rows`); the model only predicts (`predict_rows`). Two
independent scenarios run it:
**Nightly batch** (inference) and **Retrain, test and promote** (training). Each starts by setting the cut-off day
(`set_cutoff`: yesterday, or the project variable `cutoff_override`) and checking the warehouse load of that day is
finished (`check_warehouse_loaded`: a mock for now, to replace with the bank's real check); the staging recipe stops
if the postings do not reach the cut-off day.

It then trains, tests and promotes a first model, and runs the first nightly batch.

**How to use it (at the bank):** in a new empty project, import this notebook only (Notebooks > New notebook >
Import), fill in the settings below (`EDW_CONNECTION`, `CODE_ENV`, ...) and run all cells (Cell > Run All). It first
checks it can read the 2 warehouse tables, and stops with a clear message if not. It can be run again in the same
project: what exists is kept (data, saved model versions, zones, scenario history), and the recipes' code is updated.

On the test machine only: run `warehouse_simulator.ipynb` first (it writes fake tables into the local test
database), then this notebook with `EDW_CONNECTION` set to that database's connection.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast"

# ── 1. INPUT: the warehouse, read-only through SQL (nothing is ever written there) ──────────────────────────────────
# The project reads 2 tables of this schema: DIM_CUSTOMER and FCT_STATEMENT_ENTRY.
EDW_CONNECTION = "YOUR_EDW_CONNECTION"   # the Dataiku connection to the warehouse (Administration > Connections).
                                         # At the bank, for example "EDW_AI"; on the test machine "EDW_LOCAL"
EDW_SCHEMA = "EDW"                       # the schema that holds the 2 tables
HISTORY_MONTHS = 24                      # months of postings read the first time (and by the scenario Full rebuild),
                                         # up to yesterday. -1: the whole history. Each night then adds the new days
CLIENTS_LIMIT = -1                       # how many business clients to read (the lowest customer numbers): -1 all;
                                         # for example 1000 to test fast. Both: to change them later, run the setup
                                         # again, then the scenario Full rebuild

# ── 2. OUTPUT: where the project writes its own results ─────────────────────────────────────────────────────────────
OUTPUT_SQL_CONNECTION = "YOUR_SQL_CONNECTION"   # the database the forecast goes to (a connection you may write
                                                 # into, never the warehouse): the table final_forecast
OUTPUT_CONNECTION = "filesystem_managed"         # datasets (CSV files): clients, test_results, logs
OUTPUT_FOLDER_CONNECTION = "filesystem_folders"  # folders: daily_flows (Parquet), models, forecast (Parquet, by night)

# ── 3. PYTHON ───────────────────────────────────────────────────────────────────────────────────────────────────────
CODE_ENV = "YOUR_CODE_ENV"               # an existing Python code env with polars, catboost, scikit-learn and
                                         # mlflow==2.22.0 (an admin
                                         # creates it once), for example "cashflow_forecast"

## 1. Code environment

The recipes need CatBoost (the placeholder model), scikit-learn, and MLflow for Dataiku's Experiment Tracking, in an existing code env (`CODE_ENV`
above). This notebook does not create it: an admin creates it once. This checks it is there with those packages,
and otherwise stops with the request to send the admin.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())


def check_code_env(name, packages):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.9 or later) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. {request}")
    print(f"Code env {name} has {', '.join(packages)}")



# polars: the daily flows and the nightly forecast; catboost: the placeholder model (daily_catboost); scikit-learn: for
# the data scientist's models; mlflow: Experiment
# Tracking and the saved model. Dataiku tested MLflow models with mlflow 2.22.0 and 3.10.1 (DSS 14 and 15 docs, "MLflow
# models: limitations"); mlflow 3 needs Python 3.10 or later. Either works here; the request below asks for 2.22.0,
# which also runs on Python 3.9.
PACKAGES = ("polars", "catboost", "scikit-learn", "mlflow==2.22.0")
placeholders = [name for name, value in (("EDW_CONNECTION", EDW_CONNECTION), ("OUTPUT_SQL_CONNECTION",
                                                                           OUTPUT_SQL_CONNECTION), ("CODE_ENV", CODE_ENV))
                if str(value).startswith("YOUR_")]
if placeholders:
    raise RuntimeError(f"Fill in {' and '.join(placeholders)} in the settings cell above, then run again.")
check_code_env(CODE_ENV, PACKAGES)

## 2. Code

The project library gets these files, by owner:

- shared: `warehouse.py` (the warehouse tables and columns read, codes, categories, daily sums, cut-off day, warehouse check) and
  `cashflow.py` (dates, the daily flows' files, the model's input rows, the saved model name)
- training and lab: `models.py` (the model interface, the placeholder models, the test and its scores) and
  `tracking.py` (model IDs, Experiment Tracking, saved model versions, promotion rules, champion / challenger)
- inference: `forecast.py` (the production model, the nightly forecast in parallel, its SQL load)
- `forecasters.py`: the data scientist's own forecasters. Written only when it is created: running this notebook
  again never overwrites it.

Each recipe below becomes a Python recipe in the flow.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

The forecast reads 2 tables of the bank's warehouse, read-only, and creates or writes nothing there:
    DIM_CUSTOMER          EDW.DIM_CUSTOMER: the clients. The project keeps the business clients (companies and sole
                          proprietors) and only the columns below: personal fields are never read. Through the
                          database connection, a read-only SELECT of those columns (table_query), whole every run.
    FCT_STATEMENT_ENTRY   EDW.FCT_STATEMENT_ENTRY: the daily postings. The staging recipe sums them per client, day
                          and category (money in, money out) in Dataiku and adds the new days to daily_flows. Through
                          the database connection, the only query is a read-only SELECT of the columns below with a
                          date filter on VAL_DT (table_query): the last HISTORY_MONTHS months the first time (or after
                          the scenario Full rebuild), then only the days after the last day already in daily_flows.
Always through the database connection to the warehouse (the setup's EDW_CONNECTION: Oracle at the bank, read-only;
a local PostgreSQL with fake data on the test machine): the same code, only the connection differs. If a name or a
code differs at the bank, only this file changes.
"""
import numpy as np
import pandas as pd

# the tables the project reads. standard name: (table, where it is)
TABLES = {
    "clients": ("DIM_CUSTOMER", "EDW.DIM_CUSTOMER"),
    "transactions": ("FCT_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
}
STAGED = ["clients", "daily_flows", "staging_log"]       # what the staging recipe writes

# Categories: each posting gets one from its transaction type (TRNS_TYP); a type not listed goes to OTHER.
# MOCK LIST: these are the sample data's types. At the bank, replace it with the real TRNS_TYP codes (ask the
# warehouse team for the list of codes and what each means); nothing else changes.
CATEGORY_OF_TYPE = {
    "COLLECTION": "CUSTOMER_RECEIPTS", "CARD_SETTLEMENT": "CUSTOMER_RECEIPTS", "CHEQUE": "CUSTOMER_RECEIPTS",
    "SALARY": "SALARIES", "RENT": "RENT", "SUPPLIER": "SUPPLIERS", "FINANCING": "FINANCING", "GOV": "GOVERNMENT",
    "UTILITY": "UTILITIES", "CARD": "CARDS", "FEE": "FEES",
}
OTHER = "OTHER"

# the columns read. standard name: warehouse (EDW) name
COLUMNS = {
    "clients": {"client_no": "CSTMR_CIF", "customer_type": "CSTMR_TYP", "line_of_business": "LOB_DSC",
                "legal_entity": "LGL_ENTY_DSC", "sub_legal_entity": "PRTY_SUB_LGL_DSC", "id_type": "PRTY_ID_TYP_DSC",
                "segment": "RM_SGMNT_NM", "sector": "SCTR_DSC", "business_sector": "BUS_SCTR_EN_NM",
                "industry": "INDST_EN_NM", "sub_industry": "SUB_INDST_EN_NM", "city": "CITY", "status": "STS_FLG"},
    # one row per posting. AMT_LCY: SAR, negative for money out (T24). TRNS_TYP: the transaction type, which gives
    # the category. RVRSL_MRKR: set on a reversing posting, which carries the reversed posting's TRNS_RFR
    "transactions": {"entry_id": "STMT_ENTRY_ID", "client_no": "CSTMR_CIF", "account_no": "ACNT_NMBR",
                     "value_date": "VAL_DT", "direction": "CRDT_DBT_FLG", "amount": "AMT_LCY", "txn_type": "TRNS_TYP",
                     "txn_ref": "TRNS_RFR", "reversal": "RVRSL_MRKR"},
}
# Which rows of DIM_CUSTOMER are read: 2 SQL conditions, applied in the warehouse (the bank's own query, checked on
# 2026-11-08: about 420k rows, one per customer). Column names in double quotes, so they work as they are on Oracle.
# The business clients: relationship manager code starting with 12, or the sector Sole Proprietorships
CLIENTS_BUSINESS = """ "PRIME_RM" LIKE '12%' OR "SCTR_DSC" = 'Sole Proprietorships' """
# DIM_CUSTOMER keeps history (about 520 million rows): each customer's current row is the one still open
CLIENTS_CURRENT_ROW = """ "END_DT" = DATE '2199-12-31' """
# FCT_STATEMENT_ENTRY is partitioned by month on its load date AS_OF_DT (not on the value date VAL_DT). Staging
# filters on it too, so Oracle reads only the load partitions that can hold the days it needs, not the whole table:
# AS_OF_DT from POSTINGS_LOAD_MARGIN_DAYS days before the first value day read. A posting loaded after its value
# date is always read; only one loaded more than that many days BEFORE its value date would be missed.
# None: no load-date filter (every partition is read).
POSTINGS_LOAD_DATE = "AS_OF_DT"
POSTINGS_LOAD_MARGIN_DAYS = 31
ACTIVE_STATUS = ("ACTIVE", "A")               # STS_FLG of an active client; an empty STS_FLG counts as active too
                                              # (an inactive one shows INACTIVE)
SOLE_PROPRIETOR = ("SOLE", "ESTABLISHMENT")   # words of LGL_ENTY_DSC, PRTY_SUB_LGL_DSC or SCTR_DSC for a business owned by
                                              # 1 person (the sector Sole Proprietorships, for example)


def read_columns(table):
    """The warehouse columns the project reads from a table."""
    return list(COLUMNS[table].values())


def to_standard(df, table):
    """A warehouse table with the standard column names and the project's values: client numbers as text, direction
    IN / OUT; for clients, status ACTIVE / CLOSED and sole_proprietor true / false. A missing column stops here
    with its name."""
    missing = [c for c in COLUMNS[table].values() if c not in df]
    if missing:
        raise ValueError(f"{TABLES[table][0]} has no column {', '.join(missing)}: check the warehouse table")
    out = pd.DataFrame({standard: df[column] for standard, column in COLUMNS[table].items()})
    out["client_no"] = out["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "direction" in out:
        out["direction"] = out["direction"].astype(str).str.strip().str.upper().str[0].map({"C": "IN", "D": "OUT"})
    if table == "clients":
        status_raw = out["status"].astype(str).str.strip().str.upper().replace(["NAN", "NONE", "NULL", ""], "")
        active = status_raw.isin(ACTIVE_STATUS) | status_raw.eq("")      # no status at all: active too
        out["status"] = np.where(active, "ACTIVE", "CLOSED")
        legal = (out["legal_entity"].fillna("").astype(str) + " " + out["sub_legal_entity"].fillna("").astype(str)
                 + " " + out["sector"].fillna("").astype(str))
        out["sole_proprietor"] = legal.str.upper().str.contains("|".join(SOLE_PROPRIETOR))
    return out


def one_row_per_client(clients):
    """One row per client number (the last one received). The warehouse already sends one per client
    (CLIENTS_CURRENT_ROW); staging warns if it did not."""
    return clients.drop_duplicates("client_no", keep="last").reset_index(drop=True)


def value_day(dates):
    """The day of each value date, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.normalize()


def table_query(table, name, since=None, end=None, count=False, limit=None):
    """The read-only query staging runs through the database connection: only the columns read (COLUMNS) of a table,
    named in the query, so it never depends on the column list Dataiku keeps for the dataset. name: the table's full
    quoted name. For the postings, only the value dates from since (None: the whole history) to before end; the value
    date comes back as text YYYY-MM-DD (TO_CHAR: Oracle and PostgreSQL), so no time zone can move it. count: the
    same rows counted (COUNT(*)). For the clients, only the business clients' current rows (clients_filter): the
    warehouse leaves the other customers and the old versions out, so far fewer rows travel."""
    date = COLUMNS["transactions"]["value_date"] if table == "transactions" else None
    columns = [f"TO_CHAR(\"{date}\", 'YYYY-MM-DD') AS \"{date}\"" if c == date else f'"{c}"'
               for c in COLUMNS[table].values()]
    query = f"SELECT {'COUNT(*) AS N' if count else ', '.join(columns)} FROM {name}"
    if table == "clients":
        query += f" WHERE {clients_filter()}"       # business clients' current rows only: the rest stays in Oracle
        if limit:                                   # only the first `limit` business clients (CLIENTS_LIMIT)
            query += f' ORDER BY "{COLUMNS["clients"]["client_no"]}" FETCH FIRST {int(limit)} ROWS ONLY'
    if date:
        where = [f"\"{date}\" < DATE '{end:%Y-%m-%d}'"]
        if since is not None:
            where.insert(0, f"\"{date}\" >= DATE '{since:%Y-%m-%d}'")
        query += f" WHERE {' AND '.join(where)}"
    return query


def clients_filter():
    """The SQL condition on DIM_CUSTOMER: the business clients (CLIENTS_BUSINESS), only each one's current row
    (CLIENTS_CURRENT_ROW). Used for the clients and, as a subquery, for the postings."""
    current = CLIENTS_CURRENT_ROW.strip()
    return f"({CLIENTS_BUSINESS.strip()})" + (f" AND ({current})" if current else "")


def _text(value):
    return "'" + str(value).replace("'", "''") + "'"


def daily_totals_query(postings, clients, since, end, limit=None):
    """The read-only query staging runs for the postings: the warehouse itself sums them into daily totals, so only
    one row per client, day and category comes back (columns client_no, day, category, money_in, money_out,
    postings), not every posting. postings, clients: the 2 tables' full quoted names. The rules (Oracle and
    PostgreSQL; NULL-safe, as Oracle treats '' as NULL):
    - 3 steps: the selected customers (DIM_CUSTOMER with clients_filter, one row each), the postings of the
      selected days (value dates from since, None: no lower bound, to before end), then the postings joined with
      the customers; with since, also only the load dates from
      POSTINGS_LOAD_MARGIN_DAYS before it (POSTINGS_LOAD_DATE: the partitions Oracle reads);
    - without reversed postings: a posting with a reversal marker (RVRSL_MRKR) carries the reversed posting's
      reference (TRNS_RFR), and every posting with that reference is left out;
    - without transfers between the client's own accounts: on the same client and reference, a credit and a debit
      that net to zero;
    - category from the transaction type (CATEGORY_OF_TYPE, else OTHER); money_in, money_out: the credits and the
      debits as positive SAR amounts (AMT_LCY); postings: how many.
    Window functions (one pass, no NOT IN / NOT EXISTS lookups) keep it fast on a big table."""
    c = COLUMNS["transactions"]
    dates = [f"\"{c['value_date']}\" < DATE '{end:%Y-%m-%d}'"]
    if since is not None:
        dates.insert(0, f"\"{c['value_date']}\" >= DATE '{since:%Y-%m-%d}'")
        if POSTINGS_LOAD_DATE and POSTINGS_LOAD_MARGIN_DAYS is not None:     # only the load partitions needed
            loaded = since - pd.Timedelta(days=POSTINGS_LOAD_MARGIN_DAYS)
            dates.append(f"\"{POSTINGS_LOAD_DATE}\" >= DATE '{loaded:%Y-%m-%d}'")
    cif = COLUMNS["clients"]["client_no"]
    # 1. the selected customers: one row per customer (DISTINCT), so the join below can never count a posting twice
    customers = f"SELECT DISTINCT \"{cif}\" AS c_cif FROM {clients} WHERE {clients_filter()}"
    if limit:                                       # the same first `limit` business clients as table_query
        customers += f" ORDER BY c_cif FETCH FIRST {int(limit)} ROWS ONLY"
    categories = " ".join(f"WHEN {_text(t)} THEN {_text(k)}" for t, k in CATEGORY_OF_TYPE.items())
    category = f"CASE UPPER(TRIM(p_typ)) {categories} ELSE {_text(OTHER)} END"
    return f"""WITH customers AS (
  {customers}
), postings AS (
  SELECT "{c['client_no']}" AS p_cif, TO_CHAR("{c['value_date']}", 'YYYY-MM-DD') AS p_day,
         UPPER(SUBSTR(TRIM("{c['direction']}"), 1, 1)) AS p_dir, "{c['amount']}" AS p_amt,
         "{c['txn_type']}" AS p_typ, "{c['txn_ref']}" AS p_ref, "{c['reversal']}" AS p_rev, "{c['entry_id']}" AS p_id
  FROM {postings}
  WHERE {' AND '.join(dates)}
), p AS (
  SELECT postings.* FROM postings JOIN customers ON customers.c_cif = postings.p_cif
), marked AS (
  SELECT p.*, MAX(CASE WHEN p_ref IS NOT NULL AND LENGTH(TRIM(p_rev)) > 0 AND TRIM(p_rev) NOT IN ('nan', 'None')
                       THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref) AS reversed
  FROM p
), kept AS (
  SELECT marked.*,
         MAX(CASE WHEN p_dir = 'C' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif)
         + MAX(CASE WHEN p_dir = 'D' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif) AS sides,
         COALESCE(ABS(SUM(p_amt) OVER (PARTITION BY p_ref, p_cif)), 0) AS net
  FROM marked WHERE reversed = 0
)
SELECT p_cif AS "client_no", p_day AS "day", {category} AS "category",
       SUM(CASE WHEN p_dir = 'C' THEN ABS(p_amt) ELSE 0 END) AS "money_in",
       SUM(CASE WHEN p_dir = 'D' THEN ABS(p_amt) ELSE 0 END) AS "money_out",
       COUNT(p_id) AS "postings"
FROM kept
WHERE p_ref IS NULL OR NOT (sides = 2 AND net < 0.005)
GROUP BY p_cif, p_day, {category}"""


def standard_daily(df):
    """The daily totals as the warehouse returns them, with the project's types: client numbers as text, the day as
    a date, amounts and counts as numbers."""
    out = pd.DataFrame({"client_no": df["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True),
                        "day": pd.to_datetime(df["day"]),
                        "category": df["category"].astype(str),
                        "money_in": pd.to_numeric(df["money_in"]).astype(float).fillna(0.0),
                        "money_out": pd.to_numeric(df["money_out"]).astype(float).fillna(0.0),
                        "postings": pd.to_numeric(df["postings"]).astype("int64")})
    return out


def set_cutoff():
    """The forecast's cut-off: the last day of data, by default yesterday. The project variable cutoff_override
    (YYYY-MM-DD) fixes it instead: the sandbox's simulator sets it, and it reruns a past day. Sets the project
    variables cutoff_date, as_of_date and load_id (the data's as-of date), which the recipes read. The staging recipe
    stops if the warehouse has no posting up to the cut-off day.
    Run by the first step of each scenario: from warehouse import set_cutoff; set_cutoff()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    override = variables["standard"].get("cutoff_override")
    cutoff = pd.Timestamp(override) if override else pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    variables["standard"].update(cutoff_date=f"{cutoff:%Y-%m-%d}", as_of_date=f"{cutoff:%Y-%m-%d}",
                                 load_id=f"EDW_{cutoff:%Y%m%d}")
    project.set_variables(variables)
    print(f"Cut-off day {cutoff:%Y-%m-%d}" + (" (cutoff_override)" if override else " (yesterday)"))
    return f"{cutoff:%Y-%m-%d}"


def check_warehouse_loaded():
    """MOCK. Second step of each scenario: stops the scenario if the warehouse's load of the cut-off day is not
    finished (the ETL from the source systems into EDW). It always passes for now. Replace its body with the bank's
    real check, for example a read-only query on the warehouse's load-control table:
        SELECT STATUS FROM EDW.<LOAD_CONTROL_TABLE> WHERE LOAD_DATE = DATE '<cutoff_date>'   -- must say finished
    and raise an error if it is not finished. Whatever it says, staging also stops if the warehouse has no posting
    up to the cut-off day. Run as: from warehouse import check_warehouse_loaded; check_warehouse_loaded()"""
    import dataiku
    cutoff = dataiku.api_client().get_default_project().get_variables()["standard"]["cutoff_date"]
    print(f"MOCK warehouse check: the load of {cutoff} is assumed finished. Replace check_warehouse_loaded in "
          "warehouse.py with the bank's real check.")
    return True


def clear_daily_flows():
    """Step of the scenario Full rebuild (after the cut-off and the warehouse check): empties daily_flows and staging_log, so that the
    staging recipe reads the whole history again. Run as: from warehouse import clear_daily_flows; clear_daily_flows()"""
    import dataiku
    from cashflow import clear_daily
    clear_daily()
    dataiku.api_client().get_default_project().get_dataset("staging_log").clear()
    print("daily_flows and staging_log emptied: the staging recipe will read the whole history")


for _table, _columns in COLUMNS.items():          # warehouse names unique within a table
    assert len(set(_columns.values())) == len(_columns), _table
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared by every zone (project library: python/cashflow.py): dates, the business rule on amounts, and the
interfaces between the zones:
    1. the data (staging -> training, lab and inference): daily_flows (one row per client, day and category:
       money_in, money_out, postings, over each client's whole history; a folder of Parquet files, one per group of
       clients and month, see write_daily) and the dataset clients (one row per business client). Every model gets
       these and nothing else. Inference gives each model the last INPUT_DAYS days of each client (the last year),
       laid out by model_rows: one row per client, category and window, one column per day.
    2. the model (lab -> training): models.py.
    3. the saved model (training -> inference): SAVED_MODEL; inference uses its active version.
The forecast: per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 3 windows of 30 days after the cut-off day (the last day of data: yesterday): days
1-30, 31-60 and 61-90, as P10, EV and P90 in SAR."""
import logging
import warnings

import numpy as np
import pandas as pd

try:
    import polars as pl
except ImportError:         # the setup notebooks run on Dataiku's built-in Python: they only use this file's settings
    pl = None

# Quieter job logs, for every recipe (they all import this file): Dataiku's own writer prints, for each block it
# writes, pandas warnings about its own code (DataFrame is highly fragmented, is_datetime64tz_dtype is deprecated),
# and an INFO line every 10 s while it waits for the next chunk (No data to send, waiting more...). Harmless: hidden.
# Our own warnings and Dataiku's real warnings and errors still show.
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning, module=r"dataiku\.")
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"dataiku\.")
logging.getLogger("dataiku.core.dataset_write").setLevel(logging.WARNING)
# The same for each file written to a folder (Initializing folder writer, Pipe to generator thread, Sending data, Got
# end mark, ...: several lines per file, hundreds of files): Dataiku logs them on the root logger, so they are dropped
# by where they come from. Only their INFO lines: warnings and errors still show.
_CHATTY = ("dkuio.py", "write_base.py", "managed_folder.py")
logging.getLogger().addFilter(
    lambda record: record.levelno >= logging.WARNING
    or not record.pathname.replace("\\", "/").endswith(tuple(f"dataiku/core/{f}" for f in _CHATTY)))

PERIOD_DAYS = 30            # the forecast's windows, and the periods the models work with, are 30 days long
HORIZONS = (1, 2, 3)        # the windows forecast: 1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = 61-90
TOTAL = "TOTAL"             # the category of a client's all-categories net cash flow
SAVED_MODEL = "cash_forecast"
INPUT_DAYS = 365            # what a model gets for each client: its daily flows of the last 365 days up to the
                            # cut-off day (a forecaster can ask for another number: its params["input_days"])
MIN_HISTORY_DAYS = INPUT_DAYS   # a client needs this many days of history up to the cut-off to get a forecast (else
                            # status NOT_ENOUGH_HISTORY in the final forecast)
MIN_TRAINING_MONTHS = 21    # history needed to train and test a model: a year of input, the 3 windows after it, and
                            # the 6 periods of 30 days the test keeps aside (about 635 days). Less is fine to test
                            # staging only
FORECAST_COLUMNS = ["client_no", "category", "window", "p10", "ev", "p90"]   # what a forecaster returns


def _plain(df):
    """Dates as plain (time zone free) timestamps, client numbers as text."""
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    if "client_no" in df:
        df["client_no"] = df["client_no"].astype(str)
    return df


def read_dataset(name, columns=None):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps and client numbers as text."""
    import dataiku
    return _plain(dataiku.Dataset(name).get_dataframe(columns=columns))


DAILY_FLOWS = "daily_flows"     # interface 1: a managed folder of Parquet files, written by staging
DAILY_COLUMNS = ["client_no", "day", "category", "money_in", "money_out", "postings"]
DAILY_SCHEMA = dict(zip(DAILY_COLUMNS, [pl.Utf8, pl.Datetime("us"), pl.Utf8, pl.Float64, pl.Float64, pl.Int64])) if pl \
    else {}
MANIFEST = "/manifest.json"     # in the folder: the current files (which group of clients and month each holds)
CLIENTS_PER_GROUP = 5000        # clients per group when the daily flows are written from nothing


def client_groups(clients, groups):
    """The group of each client number (a Polars Series of numbers 0 .. groups - 1): a client never changes group
    (CRC32 of its number), so its rows are always in the files of the same group."""
    import zlib
    numbers = pl.Series(clients, dtype=pl.Utf8)
    unique = numbers.unique()
    group_of = dict(zip(unique.to_list(), (zlib.crc32(n.encode()) % groups for n in unique.to_list())))
    return numbers.replace_strict(group_of, return_dtype=pl.Int64)


def read_manifest(name=DAILY_FLOWS, **kwargs):
    """The folder and its manifest (None when nothing is written yet): {"groups": number of groups, "files":
    {group: {month: [path, rows]}}, "first_days": path, "rows": rows in all}. kwargs: for dataiku.Folder."""
    import json
    import dataiku
    folder = dataiku.Folder(name, **kwargs)
    if MANIFEST not in folder.list_paths_in_partition():
        return folder, None
    with folder.get_download_stream(MANIFEST) as stream:
        return folder, json.loads(stream.read())


def read_parquet(folder, path, columns=None):
    """One Parquet file of a folder, as a Polars DataFrame."""
    import io
    with folder.get_download_stream(path) as stream:
        return pl.read_parquet(io.BytesIO(stream.read()), columns=columns)


def read_group(folder, manifest, group, since=None):
    """The daily flows of a group of clients (Polars), from the day `since` on (only the files of its months are
    read), or all of them."""
    months = (manifest["files"].get(str(group)) or {})
    first = None if since is None else f"{since:%Y-%m}"
    parts = [read_parquet(folder, path) for month, (path, _) in sorted(months.items()) if first is None or month >= first]
    daily = pl.concat(parts) if parts else pl.DataFrame(schema=DAILY_SCHEMA)
    return daily if since is None else daily.filter(pl.col("day") >= since)


def read_first_days(folder, manifest):
    """Each client's first posting day: client_no, first_day (Polars)."""
    return read_parquet(folder, manifest["first_days"])


def read_daily(name=DAILY_FLOWS, columns=None, **kwargs):
    """All of daily_flows as one pandas DataFrame (for training, the lab, the sample data; the nightly forecast reads
    one group of clients at a time, read_group)."""
    folder, manifest = read_manifest(name, **kwargs)
    files = [path for months in (manifest or {}).get("files", {}).values() for path, _ in months.values()]
    daily = pl.concat([read_parquet(folder, p, columns) for p in files]) if files else pl.DataFrame(schema=DAILY_SCHEMA)
    return _plain(daily.to_pandas())


def write_daily(parts, groups, name=DAILY_FLOWS, base=None):
    """Write daily flows: each part (pandas or Polars, DAILY_COLUMNS) holds rows of some groups of clients; its rows of
    each group and calendar month go to one Parquet file, group_0007/2026-11_<run>.parquet, added to the rows already
    in that file (base: the current manifest, when adding new days; None: from nothing). Files are never changed in
    place: new ones are written, then the manifest points to them (a failed run leaves the daily flows as they
    were), then the files it no longer lists are removed. Also each client's first posting day (first_days). Returns
    the new manifest."""
    import io
    import json
    import uuid
    import dataiku
    folder = dataiku.Folder(name)
    run = f"{pd.Timestamp.now():%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
    files = {g: dict(months) for g, months in (base or {}).get("files", {}).items()}
    firsts = [read_first_days(folder, base)] if base else []

    def write(path, frame):
        buffer = io.BytesIO()
        frame.write_parquet(buffer)
        with folder.get_writer(path) as writer:
            writer.write(buffer.getvalue())

    for part in parts:
        part = (part if isinstance(part, pl.DataFrame) else pl.from_pandas(part[DAILY_COLUMNS])).select(
            [pl.col(c).cast(k) for c, k in DAILY_SCHEMA.items()])
        if part.is_empty():
            continue
        firsts.append(part.group_by("client_no").agg(first_day=pl.col("day").min()))
        part = part.with_columns(group=client_groups(part["client_no"], groups), month=pl.col("day").dt.strftime("%Y-%m"))
        for (group, month), rows in part.group_by(["group", "month"]):
            group, rows = str(group), rows.drop(["group", "month"])
            before = files.get(group, {}).get(month)
            if before:
                rows = pl.concat([read_parquet(folder, before[0]), rows])
            path = f"/group_{int(group):04d}/{month}_{run}.parquet"
            write(path, rows.sort(["client_no", "day", "category"]))
            files.setdefault(group, {})[month] = [path, len(rows)]
    first_days = f"/first_days_{run}.parquet"
    write(first_days, pl.concat(firsts).group_by("client_no").agg(pl.col("first_day").min()).sort("client_no")
          if firsts else pl.DataFrame(schema={"client_no": pl.Utf8, "first_day": pl.Datetime("us")}))
    manifest = {"groups": groups, "files": files, "first_days": first_days,
                "rows": sum(rows for months in files.values() for _, rows in months.values())}
    with folder.get_writer(MANIFEST) as writer:
        writer.write(json.dumps(manifest).encode())
    kept = {first_days, MANIFEST} | {path for months in files.values() for path, _ in months.values()}
    for path in folder.list_paths_in_partition():
        if path not in kept:
            folder.delete_path(path)
    return manifest


def clear_daily(name=DAILY_FLOWS):
    """Remove every file of daily_flows (the scenario Full rebuild)."""
    import dataiku
    folder = dataiku.Folder(name)
    for path in folder.list_paths_in_partition():
        folder.delete_path(path)


def peak_memory_mb():
    """The most memory this process used, MB (Linux: VmHWM)."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmHWM:"):
                    return round(int(line.split()[1]) / 1024)
    except OSError:
        pass
    import resource
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)


def append_row(name, columns, row):
    """Add one row to a log dataset (an appended output of the recipe) with fixed column types, so that rows of
    failed runs (with empty values) never change how the earlier rows are read. columns: {name: Dataiku type}, one of
    string, bigint, double, boolean. When the columns changed since the last row (a new version of the code), the
    earlier rows are first read with the old columns and written again with the new ones, by name (a new column is
    empty in them), so they never end up under the wrong column."""
    import dataiku
    kinds = {"string": "string", "bigint": "Int64", "double": "float64", "boolean": "boolean"}
    frame = pd.DataFrame([{c: row.get(c) for c in columns}])
    dataset = dataiku.Dataset(name)
    before = [c["name"] for c in dataset.read_schema(raise_if_empty=False)]
    if before and before != list(columns):
        earlier = dataiku.Dataset(name, ignore_flow=True).get_dataframe()      # read with the old columns
        print(f"{name}: its columns changed; its {len(earlier)} earlier rows are written again with the new columns")
        frame = pd.concat([earlier.reindex(columns=list(columns)), frame], ignore_index=True)
        dataset.spec_item["appendMode"] = False                                 # this time: all the rows again
    for c, kind in columns.items():
        frame[c] = (frame[c].where(frame[c].notna(), "").astype(str) if kind == "string"
                    else pd.to_numeric(frame[c], errors="coerce").round().astype(kinds[kind]) if kind == "bigint"
                    else pd.to_numeric(frame[c], errors="coerce").astype(kinds[kind]) if kind == "double"
                    else frame[c].astype("boolean"))
    dataset.write_schema([{"name": c, "type": kind} for c, kind in columns.items()])
    with dataset.get_writer() as writer:
        writer.write_dataframe(frame)


class Progress:
    """Progress lines in the job's log, for long steps: how much is done (of how much, when known), the percentage,
    the time so far and about how long is left. One line at most every `every` seconds, and the last one always. They
    start with PROGRESS, so the setup notebook can show them while it waits for the job.
        progress = Progress("reading postings", total=41_000_000); progress.add(200_000) ... progress.finish()"""

    def __init__(self, what, total=None, every=30):
        import time
        self.what, self.total, self.every = what, total, every
        self.started = self.printed = time.time()
        self.done = 0
        print(f"PROGRESS {what}: starting" + (f", {total:,} to do" if total else ""), flush=True)

    def add(self, n=1):
        import time
        self.done += n
        if time.time() - self.printed >= self.every:
            self.show()

    def finish(self):
        self.show(last=True)

    def show(self, last=False):
        import time
        now = time.time()
        self.printed = now
        minutes = (now - self.started) / 60
        if last:
            print(f"PROGRESS {self.what}: done, {self.done:,} in {minutes:.1f} min", flush=True)
            return
        line = f"PROGRESS {self.what}: {self.done:,}"
        if self.total:
            share = min(self.done / self.total, 1.0)
            line += f" of {self.total:,} ({share:.0%})"
        line += f", {minutes:.1f} min so far"
        if self.total and 0 < share < 1:
            line += f", about {minutes * (1 - share) / share:.0f} min left"
        print(line, flush=True)


def waiting(what, every=60):
    """While the warehouse runs a query and no row has come back yet, a PROGRESS line every `every` seconds, so a
    long query never looks stuck. Returns an Event: set it when the first row arrives (or the query ends)."""
    import threading
    import time
    stop, started = threading.Event(), time.time()

    def beat():
        while not stop.wait(every):
            print(f"PROGRESS {what}: no row yet, the warehouse is still running the query "
                  f"({(time.time() - started) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    return stop


class Spread:
    """Rows spread over n files on local disk by the hash of a column, then read back one file at a time: all the
    rows with the same value of that column end up in the same file, and memory holds one file, not the whole table.
    Used by staging (postings by reference, then daily flows by client)."""

    def __init__(self, folder, n):
        import os
        os.makedirs(folder, exist_ok=True)
        self.n = max(1, int(n))
        self.paths = [os.path.join(folder, f"part_{i}.pkl") for i in range(self.n)]

    def add(self, df, key):
        """key: a column name, or a Series (same index as df) to spread by."""
        import pickle
        key = df[key] if isinstance(key, str) else key
        part = pd.util.hash_pandas_object(key.astype(str), index=False).to_numpy() % self.n
        for i, rows in df.groupby(part):
            with open(self.paths[i], "ab") as f:
                pickle.dump(rows, f, protocol=4)

    def parts(self):
        """Each file's rows (a DataFrame); files that got no rows are skipped."""
        import os
        import pickle
        for path in self.paths:
            if not os.path.exists(path):
                continue
            frames = []
            with open(path, "rb") as f:
                while True:
                    try:
                        frames.append(pickle.load(f))
                    except EOFError:
                        break
            yield pd.concat(frames, ignore_index=True)


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def cutoff_date():
    """The forecast's cut-off: the last day of data, by default yesterday (project variable cutoff_date, set by the
    first step of each scenario: warehouse.set_cutoff)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_date"])


def whole_number(name, value):
    """A setting that is a whole number, or None for "all" (empty, None, -1 or "all"). Stops with a clear message
    otherwise."""
    value = "" if value is None else str(value).strip()
    if value.lower() in ("", "none", "all", "-1"):
        return None
    try:
        number = float(value)
    except ValueError:
        number = 0.5
    if number != int(number) or number < 1:
        raise ValueError(f"{name} is {value!r}: it must be a whole number (1 or more), or -1 / None for all")
    return int(number)


def history_start(cutoff):
    """The first day staging reads when it reads everything (the first run, or the scenario Full rebuild): the
    project variable history_months months before the cut-off day, or None for the whole history. With less than
    about 18 months, few clients reach the history a forecast and the model test need (MIN_HISTORY_DAYS, and the
    last 6 periods of 30 days kept aside): fine to test the pipeline, not to judge a model."""
    import dataiku
    months = whole_number("history_months", dataiku.get_custom_variables().get("history_months"))
    return None if months is None else cutoff - pd.DateOffset(months=months) + pd.Timedelta(days=1)


def clients_limit():
    """How many business clients staging reads (project variable clients_limit): None for all of them."""
    import dataiku
    return whole_number("clients_limit", dataiku.get_custom_variables().get("clients_limit"))


def until(daily, cutoff):
    """The daily flows known at the cut-off: the days up to the cut-off day (included)."""
    return daily[pd.to_datetime(daily["day"]) <= cutoff]


def window(cutoff, h):
    """The first and last day of window h after the cut-off day (h = 1: days 1-30)."""
    return cutoff + pd.Timedelta(days=PERIOD_DAYS * (h - 1) + 1), cutoff + pd.Timedelta(days=PERIOD_DAYS * h)


def finish(ev, p10, p90, index=None):
    """P10 <= EV <= P90 (SAR). The net cash flow can be below zero."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    return pd.DataFrame({"p10": np.minimum(p10, ev), "ev": ev, "p90": np.maximum(p90, ev)}, index=index).round(2)


def _sums(d, key):
    """Net cash flow per client, category (and TOTAL) and key."""
    by_category = d.groupby(["client_no", "category", key])["net"].sum()
    total = d.groupby(["client_no", key])["net"].sum().to_frame().assign(category=TOTAL)
    total = total.set_index("category", append=True).reorder_levels(["client_no", "category", key])["net"]
    return pd.concat([by_category, total])


def _wide(sums, columns):
    """One row per client and category, the given columns; 0 after a series' first value, empty before it."""
    if sums.empty:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=["client_no", "category"]))
    wide = sums.unstack(sums.index.names[-1]).reindex(columns=columns)
    started = wide.notna().cumsum(axis=1) > 0
    return wide.fillna(0.0).where(started).sort_index()


def period_matrix(daily, cutoff):
    """The net cash flow (money in minus money out) per 30-day period ending at the cut-off day: one row per client
    and category (and per client in TOTAL), one column per period, named by its last day (the last column is the 30
    days up to the cut-off, the one before the 30 days before, ...). 0 for a period with no posting after the
    series' first period, empty before it. The window forecast h is the period ending cutoff + 30 x h days."""
    d = until(daily, cutoff)
    back = (cutoff - pd.to_datetime(d["day"])).dt.days // PERIOD_DAYS
    d = d.assign(period=(cutoff - pd.to_timedelta(back * PERIOD_DAYS, unit="D")).values,
                 net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return _wide(pd.Series(dtype=float), [])
    oldest = int(back.max())
    return _wide(_sums(d, "period"), [cutoff - pd.Timedelta(days=PERIOD_DAYS * k) for k in range(oldest, -1, -1)])


def period_series(daily, cutoff):
    """period_matrix as rows: client_no, category, period (its last day), net (no row before a series' first)."""
    long = period_matrix(daily, cutoff).reset_index().melt(id_vars=["client_no", "category"], var_name="period",
                                                           value_name="net")
    long["period"] = pd.to_datetime(long["period"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "period"]).reset_index(drop=True)


def monthly_series(daily, cutoff):
    """For the EDA: the net cash flow per client, category (and TOTAL) and calendar month, up to the cut-off day (its
    month is partial): client_no, category, month, net (no row before a series' first month)."""
    d = until(daily, cutoff)
    d = d.assign(month=month_start(d["day"]).values, net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return pd.DataFrame(columns=["client_no", "category", "month", "net"])
    months = list(pd.date_range(d["month"].min(), month_start([cutoff])[0], freq="MS"))
    long = _wide(_sums(d, "month"), months).reset_index().melt(id_vars=["client_no", "category"], var_name="month",
                                                                value_name="net")
    long["month"] = pd.to_datetime(long["month"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "month"]).reset_index(drop=True)


def input_days(model=None):
    """How many days of daily flows a model gets for each client: its params["input_days"], else INPUT_DAYS."""
    return int((getattr(model, "params", None) or {}).get("input_days") or INPUT_DAYS)


def day_columns(days=INPUT_DAYS):
    """The names of the model's day columns, the oldest first: day_365, ..., day_1 (day_1: the cut-off day)."""
    return [f"day_{days - i}" for i in range(days)]


def day_matrix(daily, cutoff, days=INPUT_DAYS):
    """The net cash flow (money in minus money out) of each of the last `days` days up to the cut-off day, per client
    and category (and per client in TOTAL: all its categories together); 0 for a day with no posting. Returns the
    keys (Polars: client_no, category, sorted) and the matrix (numpy float32, one row per key, the oldest day first).
    daily: Polars or pandas."""
    d = daily if isinstance(daily, pl.DataFrame) else pl.from_pandas(daily[DAILY_COLUMNS])
    end = pd.Timestamp(cutoff).to_pydatetime()
    d = d.filter(pl.col("day").is_between(end - pd.Timedelta(days=days - 1), end)).select(
        pl.col("client_no").cast(pl.Utf8), pl.col("category").cast(pl.Utf8),
        (days - 1 - (pl.lit(end) - pl.col("day")).dt.total_days()).alias("column"),
        (pl.col("money_in") - pl.col("money_out")).alias("net"))
    d = pl.concat([d, d.with_columns(category=pl.lit(TOTAL))])
    keys = d.select("client_no", "category").unique().sort("client_no", "category")
    d = d.join(keys.with_row_index("series"), on=["client_no", "category"])
    flat = d["series"].to_numpy().astype(np.int64) * days + d["column"].to_numpy()
    matrix = np.bincount(flat, weights=d["net"].to_numpy(), minlength=len(keys) * days)
    return keys, matrix.reshape(len(keys), days).astype(np.float32)


def model_rows(keys, matrix):
    """The model's input (interface 2): one row per client, category and window (1, 2, 3: days 1-30, 31-60, 61-90
    after the cut-off day), its columns client_no, category, window, day_365 ... day_1 (the net cash flow of each
    day, SAR). keys, matrix: from day_matrix. Polars."""
    days = pl.DataFrame(matrix, schema=day_columns(matrix.shape[1]), orient="row")
    return pl.concat([keys.with_columns(window=pl.lit(h, pl.Int8)).hstack(days) for h in HORIZONS])
'''

In [ ]:
MODELS_CODE = r'''
"""Models (project library: python/models.py): interface 2, the model, the placeholder models and the test every
model goes through. Training (train_and_test, champion_challenger) and the lab use it; the nightly forecast only gives
the production model its input rows (forecast.py).

A model (a "forecaster" in the project variable that picks it) is a class with:
    name            its name. The project variable forecaster says which one the scenario Retrain, test and promote
                    trains (default: daily_catboost).
    params          its settings (a dict), recorded in Experiment Tracking. params["input_days"]: how many days of
                    each client it gets (default 365).
    fit(daily, clients, cutoff)
                    learns from the data known at the cut-off day (interface 1, cashflow.py): daily: daily_flows
                    (client_no, day, category, money_in, money_out, postings, pandas), every day up to the cut-off
                    day; clients: the clients dataset; cutoff: the cut-off day (a Timestamp: the last day of data).
    predict_rows(rows)
                    what the nightly forecast calls. rows: a Polars DataFrame (cashflow.model_rows), one row per
                    client, category (each category the client has, and TOTAL: all its categories together) and window
                    (1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = days 61-90): client_no, category, window,
                    day_365 ... day_1 (the net cash flow of each of the last input_days days, day_1 the cut-off day).
                    Returns a numpy array, one row per input row: P10, EV, P90 of the window's net cash flow (SAR).
    predict(daily, clients, cutoff)
                    the same from daily flows, for the test, champion / challenger and the lab: DayModel gives it.
The data scientist's own models go in forecasters.py: the setup notebook creates that file once and never overwrites
it. lab_compare (notebook forecast_lab) tries them against the production model without touching production.
"""
import numpy as np
import pandas as pd
import polars as pl

from cashflow import (FORECAST_COLUMNS, HORIZONS, INPUT_DAYS, PERIOD_DAYS, cutoff_date, day_matrix, finish,
                      model_rows, period_matrix, period_series, until)

HOLDOUT = 6                 # the last 6 periods of 30 days are kept aside to test the models
OUTPUT = ["p10", "ev", "p90"]


class DayModel:
    """Base of the placeholders: predict (from daily flows) through predict_rows (from the model's input rows), so the
    test and the nightly forecast run the same code."""
    params = {"input_days": INPUT_DAYS}

    def predict(self, daily, clients, cutoff):
        rows = model_rows(*day_matrix(until(daily, cutoff), cutoff, self.params["input_days"]))
        if rows.is_empty():
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        values = self.predict_rows(rows)
        out = rows.select("client_no", "category", "window").to_pandas()
        out["window"] = out["window"].astype(int)
        out[OUTPUT] = values
        return out[FORECAST_COLUMNS]

    @staticmethod
    def days(rows):
        """The day columns of the input rows, as a numpy float32 array (the oldest day first)."""
        return rows.select(pl.col("^day_.*$")).to_numpy().astype(np.float32, copy=False)


class RecentAverage(DayModel):
    """A simple model with no machine learning, to compare others with (and fall back on). Per client and category,
    EV = the average net cash flow of the last 3 periods of 30 days (the same for the 3 windows). P10 and P90 = EV -/+
    width x the series' standard deviation over the last 12 periods, where fit sets width so that, over the 12
    periods before the cut-off, 80% of the actual periods fell inside."""
    name = "recent_average"
    params = {"input_days": INPUT_DAYS, "periods": 3, "spread_periods": 12, "coverage": 0.8}

    def fit(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        n, s = self.params["periods"], self.params["spread_periods"]
        z = []
        for k in range(1, min(12, wide.shape[1] - 1) + 1):         # each of the last 12 periods, from the ones before
            past, actual = wide.iloc[:, :-k], wide.iloc[:, -k]
            ev, sd = past.iloc[:, -n:].mean(axis=1), past.iloc[:, -s:].std(axis=1)
            ok = actual.notna() & ev.notna() & (sd > 0)
            z.append(((actual - ev).abs() / sd)[ok])
        z = pd.concat(z) if z else pd.Series(dtype=float)
        self.width = float(z.quantile(self.params["coverage"])) if len(z) else 1.28
        return self

    def predict_rows(self, rows):
        s = self.params["spread_periods"]
        days = self.days(rows)[:, -PERIOD_DAYS * s:]
        periods = days.reshape(len(days), s, PERIOD_DAYS).sum(axis=2)     # the last 12 periods of 30 days
        ev = periods[:, -self.params["periods"]:].mean(axis=1)
        sd = periods.std(axis=1, ddof=1)
        return np.column_stack([ev - self.width * sd, ev, ev + self.width * sd])


class DailyCatBoost(DayModel):
    """The production placeholder (option A), until the data scientist's models replace it: ONE CatBoost model with
    the MultiQuantile loss gives P10, EV (the median, far less thrown off by a few huge days than the mean) and P90
    together, for every category, TOTAL and the 3 windows. Its inputs are the rows of cashflow.model_rows: the 365
    days (divided by the series' scale so small and large clients look alike), the category (as a number) and the
    window. Nothing else: no sums over 30 days, no calendar.
    Target in training: the net cash flow of the window (the sum of its 30 days), divided by the same scale. Trained on
    rolling cut-offs: every 30 days back from the cut-off (up to `cutoffs` of them), each series whose whole year of
    input is known, each window whose 30 days are known; at most max_clients clients and max_rows rows (random
    samples beyond that, the same every run). threads: CatBoost's threads when predicting (the nightly forecast sets
    it from its workers)."""
    name = "daily_catboost"
    params = {"input_days": INPUT_DAYS, "cutoffs": 24, "max_clients": 20_000, "max_rows": 200_000,
              "iterations": 300, "depth": 6, "learning_rate": 0.1}
    QUANTILES = (0.1, 0.5, 0.9)          # the model's 3 outputs, in this order: P10, EV, P90
    threads = -1

    def new_model(self):
        from catboost import CatBoostRegressor
        alphas = ",".join(str(q) for q in self.QUANTILES)
        return CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", iterations=self.params["iterations"],
                                 depth=self.params["depth"], learning_rate=self.params["learning_rate"],
                                 random_seed=0, verbose=False, allow_writing_files=False)

    def _inputs(self, days, codes, window):
        """The model's inputs, and each row's scale."""
        scale = np.maximum(np.abs(days).mean(axis=1) * PERIOD_DAYS, 100.0).astype(np.float32)
        return np.column_stack([days / scale[:, None], codes, window]).astype(np.float32), scale

    def _codes(self, categories):
        """Each category as a number (its place among the categories seen in training; empty if new)."""
        code = {c: i for i, c in enumerate(self.categories)}
        return pl.Series(categories, dtype=pl.Utf8).replace_strict(code, default=None, return_dtype=pl.Float32)\
            .fill_null(np.nan).to_numpy()

    def fit(self, daily, clients, cutoff):
        days, rng = self.params["input_days"], np.random.default_rng(0)
        known = until(daily, cutoff)
        if known.empty:
            raise ValueError(f"{self.name}: no daily flows up to {cutoff:%Y-%m-%d} to train on")
        numbers = known["client_no"].astype(str).unique()
        if len(numbers) > self.params["max_clients"]:
            numbers = rng.choice(np.sort(numbers), self.params["max_clients"], replace=False)
            known = known[known["client_no"].astype(str).isin(set(numbers))]
        first = pd.to_datetime(known.groupby("client_no")["day"].min())
        span = (cutoff - first.min()).days + 1
        keys, matrix = day_matrix(known, cutoff, span)                   # every day, the oldest first
        self.categories = sorted(set(keys["category"]))
        codes = self._codes(keys["category"])
        start = span - 1 - (cutoff - first.reindex(keys["client_no"].to_list())).dt.days.to_numpy()
        blocks = []                      # (the cut-off's column, window, the series whose input year is known)
        for k in range(1, self.params["cutoffs"] + 1):
            end = span - 1 - PERIOD_DAYS * k
            if end - days + 1 < 0:
                break
            rows = np.flatnonzero(start <= end - days + 1)
            blocks += [(end, h, rows) for h in HORIZONS if k >= h and len(rows)]
        total = sum(len(rows) for _, _, rows in blocks)
        if not total:
            raise ValueError(f"{self.name}: not enough history to train: a series needs {days} days of input and the "
                             f"{PERIOD_DAYS} days after them (daily flows from {first.min():%Y-%m-%d} to "
                             f"{cutoff:%Y-%m-%d})")
        keep = min(1.0, self.params["max_rows"] / total)
        Xs, ys = [], []
        for end, h, rows in blocks:
            rows = rows[rng.random(len(rows)) < keep]
            X, scale = self._inputs(matrix[rows, end - days + 1:end + 1], codes[rows], np.full(len(rows), h))
            Xs.append(X)
            ys.append(matrix[rows, end + PERIOD_DAYS * (h - 1) + 1:end + PERIOD_DAYS * h + 1].sum(axis=1) / scale)
        self.model = self.new_model().fit(np.vstack(Xs), np.concatenate(ys))
        return self

    def predict_rows(self, rows):
        X, scale = self._inputs(self.days(rows), self._codes(rows["category"]), rows["window"].to_numpy())
        return self.model.predict(X, thread_count=self.threads) * scale[:, None]


def known_forecasters():
    """The forecasters Retrain can train, by name: the placeholders (daily_catboost, recent_average) and those
    listed in FORECASTERS in forecasters.py. Also returns the problem reading forecasters.py, if any (the placeholders
    still work)."""
    found, problem = {DailyCatBoost.name: DailyCatBoost, RecentAverage.name: RecentAverage}, None
    try:
        import forecasters
        for forecaster in forecasters.FORECASTERS:
            found[forecaster.name] = forecaster
    except Exception as e:
        problem = f"forecasters.py could not be read ({type(e).__name__}: {e})"
    return found, problem


def new_forecaster(name=None):
    """A new, untrained forecaster: the one named by the project variable forecaster (default daily_catboost)."""
    import dataiku
    name = name or dataiku.get_custom_variables().get("forecaster") or DailyCatBoost.name
    found, problem = known_forecasters()
    if name not in found:
        raise ValueError(f"No forecaster named {name!r} (project variable forecaster). Known: {', '.join(found)}. "
                         + (problem if problem else "Add yours to FORECASTERS in forecasters.py."))
    forecaster = found[name]()
    missing = [a for a in ("name", "fit", "predict") if not hasattr(forecaster, a)]
    if missing:
        raise ValueError(f"Forecaster {name} has no {', '.join(missing)} (see models.py)")
    return forecaster


def forecast(forecaster, daily, clients, cutoff):
    """P10, EV and P90 of a trained forecaster (SAR), checked, with P10 <= EV <= P90: one row per client, category and
    window. Everything that predicts goes through here: the test, champion / challenger, the lab and the
    nightly forecast."""
    out = forecaster.predict(daily, clients, cutoff)
    name = getattr(forecaster, "name", type(forecaster).__name__)
    if not isinstance(out, pd.DataFrame) or not set(FORECAST_COLUMNS) <= set(out.columns):
        raise ValueError(f"Forecaster {name}: predict must return a DataFrame with the columns "
                         f"{', '.join(FORECAST_COLUMNS)}")
    empty = [c for c in OUTPUT if out[c].isna().any()]
    if empty:
        raise ValueError(f"Forecaster {name}: predict returned empty values in {', '.join(empty)}")
    out = out[FORECAST_COLUMNS].reset_index(drop=True)
    out["client_no"] = out["client_no"].astype(str)
    out["window"] = out["window"].astype(int)
    out[OUTPUT] = finish(out["ev"], out["p10"], out["p90"], out.index)
    return out.sort_values(["client_no", "category", "window"]).reset_index(drop=True)


def active(daily, clients):
    """The active clients and their daily flows: the clients the test and the forecast are made for."""
    clients = clients[clients["status"] == "ACTIVE"]
    return daily[daily["client_no"].isin(set(clients["client_no"]))], clients


def test_forecasts(fitted, daily, clients, cutoff):
    """The test, the same for every model: a forecaster trained without the last 6 periods of 30 days forecasts the
    windows after each test cut-off (the 6 days, 30 days apart, before the cut-off) from the data known at that day,
    for the active clients. Returns one row per client, category, test cut-off and window whose days are known:
    target (the actual net cash flow), p10, ev, p90 and baseline (the simple guess: the same as the 30 days before the
    test cut-off)."""
    daily, clients = active(daily, clients)
    known = period_series(daily, cutoff)
    actual = known.rename(columns={"period": "target_end", "net": "target"})
    last = known.rename(columns={"period": "test_cutoff", "net": "baseline"})
    parts = []
    for k in range(HOLDOUT):
        t = cutoff - pd.Timedelta(days=PERIOD_DAYS * (HOLDOUT - k))
        f = forecast(fitted, until(daily, t), clients, t).assign(test_cutoff=t)
        f["target_end"] = f["window"].map({h: t + pd.Timedelta(days=PERIOD_DAYS * h) for h in HORIZONS})
        parts.append(f[f["target_end"] <= cutoff])
    keys = ["client_no", "category"]
    tested = (pd.concat(parts, ignore_index=True)
              .merge(actual, on=keys + ["target_end"], how="left")
              .merge(last, on=keys + ["test_cutoff"], how="left"))
    tested[["target", "baseline"]] = tested[["target", "baseline"]].fillna(0.0)
    return tested


def backtest(forecaster, daily, clients, cutoff):
    """Train the forecaster without the last 6 periods of 30 days (the data known at the first test cut-off), then
    test it on them (test_forecasts). Returns the test rows; the forecaster is now the backtest forecaster."""
    first = cutoff - pd.Timedelta(days=PERIOD_DAYS * HOLDOUT)
    forecaster.fit(until(daily, first), clients, first)
    return test_forecasts(forecaster, daily, clients, cutoff)


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def test_results(tested):
    """Test scores per window (1, 2, 3) and overall (window ALL), for the TOTAL rows (level TOTAL) and the
    category rows (level CATEGORIES). The overall scores, which promotion uses, are on all rows."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    level = np.where(tested["category"] == "TOTAL", "TOTAL", "CATEGORIES")
    results = pd.concat([
        tested.assign(level=level).groupby(["level", "window"])[scored].apply(scores).reset_index(),
        tested.assign(level=level).groupby("level")[scored].apply(scores).reset_index().assign(window="ALL"),
        scores(tested).to_frame().T.assign(level="ALL", window="ALL")], ignore_index=True)
    results["window"] = results["window"].astype(str)
    return results[["level", "window"] + [c for c in results if c not in ("level", "window")]]


def run_metrics(results):
    """The scores recorded in Experiment Tracking: overall, and the EV error of the TOTAL rows per window.
    results: from test_results."""
    overall = results[(results["level"] == "ALL")].iloc[0]
    metrics = {k: overall[c] for k, c in (("wape", "wape_model"), ("wape_baseline", "wape_baseline"),
                                          ("coverage_p10_p90", "coverage_p10_p90"), ("below_p10", "below_p10"),
                                          ("above_p90", "above_p90"))}
    for row in results[(results["level"] == "TOTAL") & (results["window"] != "ALL")].itertuples():
        metrics[f"wape_total_w{row.window}"] = row.wape_model
    return metrics


def _lab_test():
    """The lab's data and the production model's test, the same as champion / challenger's: daily flows, clients,
    the production model's ID and its backtest forecaster's test rows (None without a production model)."""
    import tracking
    from cashflow import read_daily, read_dataset
    daily, clients = read_daily(), read_dataset("clients")
    production_id = tracking.production_model_id()
    production = (test_forecasts(tracking.load_backtest(production_id), daily, clients, cutoff_date())
                  if production_id else None)
    return daily, clients, production_id, production


def _by_window(tested):
    """Overall test scores per window (all rows), indexed by window (1, 2, 3, ALL)."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    per = tested.groupby("window")[scored].apply(scores)
    per.index = per.index.astype(str)
    per.loc["ALL"] = scores(tested)
    return per


def _record_try(forecaster, tested, production_id):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab). Returns the run ID."""
    import tracking
    params = {"forecaster": forecaster.name, **getattr(forecaster, "params", {}),
              "cutoff_date": f"{cutoff_date():%Y-%m-%d}", "test_rows": len(tested),
              "compared_with": production_id or "none"}
    run_name = f"{forecaster.name}_{pd.Timestamp.now():%Y%m%dT%H%M}"
    return tracking.log_lab_run(run_name, params, run_metrics(test_results(tested)), forecaster)


def lab_compare(forecasters, record=True):
    """Try several forecasters at once against the production model, on the same test as champion / challenger,
    without touching production. Each forecaster is trained without the last 6 periods of 30 days; each try is
    recorded in Experiment Tracking (experiment cashflow_lab) unless record is False. Returns a dict:
        board       one row per model (production first, simple guess last): EV error, P10-P90 coverage, share below
                    P10 and above P90, the decision champion / challenger would take and its reason, the lab run ID
        errors      EV error per window: the simple guess, the production model and each forecaster
        coverage    P10-P90 coverage per window: the production model and each forecaster
        tested      the test rows with each model's p10, ev, p90, by name, for your own analysis"""
    import tracking
    names = [f.name for f in forecasters]
    if len(set(names)) != len(names) or {"production", "simple_guess_baseline"} & set(names):
        raise ValueError(f"Give each forecaster its own name: {names}")
    daily, clients, production_id, production = _lab_test()
    tested = {} if production is None else {"production": production}
    for forecaster in forecasters:
        tested[forecaster.name] = backtest(forecaster, daily, clients, cutoff_date())
    run_ids = {f.name: _record_try(f, tested[f.name], production_id) for f in forecasters} if record else {}
    reference = scores(production) if production is not None else None
    board = []
    for name, rows in tested.items():
        s = scores(rows)
        decision, reason = (("", f"the production model, {production_id}") if name == "production"
                            else tracking.decide(s, reference))
        board.append({"model": name, "ev_error": s["wape_model"], "coverage_p10_p90": s["coverage_p10_p90"],
                      "below_p10": s["below_p10"], "above_p90": s["above_p90"], "decision": decision,
                      "reason": reason, "lab_run_id": run_ids.get(name, "")})
    board.append({"model": "simple_guess_baseline", "ev_error": scores(next(iter(tested.values())))["wape_baseline"],
                  "decision": "", "reason": "the same as the 30 days before: the bar to beat", "lab_run_id": ""})
    board = pd.DataFrame(board).set_index("model")
    per = {name: _by_window(rows) for name, rows in tested.items()}
    first = next(iter(per.values()))
    errors = pd.DataFrame({"simple_guess_baseline": first["wape_baseline"],
                           **{name: found["wape_model"] for name, found in per.items()}})
    coverage = pd.DataFrame({name: found["coverage_p10_p90"] for name, found in per.items()})
    return {"board": board, "errors": errors, "coverage": coverage, "tested": tested}
'''

In [ ]:
TRACKING_CODE = r'''
"""Model tracking and promotion (project library: python/tracking.py), for training and the lab.

Every training run is one model with its own model ID, recorded in Dataiku's Experiment Tracking (MLflow; artifacts in
the managed folder "experiments"): its forecaster and settings, the cut-off, test scores on the last 6 periods of
30 days, the backtest forecaster (trained without those periods, for champion / challenger tests) and one MLflow model (the final
forecaster, which returns P10, EV and P90). It is deployed as a new version (version ID = model ID) of the saved model
cash_forecast (interface 3, cashflow.SAVED_MODEL). The production model is the active version; promoting a model
activates it, and the nightly forecast always uses the active version.
Lab tries (notebook forecast_lab) go to their own experiment, cashflow_lab, in the same folder. They are never
deployed.
"""
import inspect
import os
import pickle
import sys
import tempfile

import pandas as pd

from cashflow import SAVED_MODEL

EXPERIMENT, FOLDER = "cashflow_forecast", "experiments"
LAB_EXPERIMENT, LAB_FOLDER = "cashflow_lab", FOLDER        # lab tries: their own experiment, same folder
# The promotion rules: project variables (change them in the project's variables, no code change), with these
# defaults. A candidate is promoted only if all hold:
PROMOTION_RULES = {
    "promotion_min_coverage": 0.70,      # its P10-P90 coverage is at least this...
    "promotion_max_coverage": 0.90,      # ...and at most this (80% is the aim)
    "promotion_beat_simple_guess": True,  # its EV error is lower than the simple guess's (same as the month before)
    "promotion_min_gain": 0.0,           # its EV error is lower than the production model's by more than this share
                                         # (0.02: at least 2% lower). A tie keeps the production model.
}


def _project():
    import dataiku
    return dataiku.api_client().get_default_project()


def _mlflow(folder):
    """Experiment Tracking in the folder. By its ID: outside a recipe (the lab notebook), setup_mlflow does not resolve
    a folder name."""
    import dataiku
    return _project().setup_mlflow(managed_folder=dataiku.Folder(folder).get_id())


def _experiment_name(mlflow, experiment, folder):
    """The experiment, kept in the folder. If an experiment of that name points to another folder (the folder was
    deleted and created again, for example when the flow was deleted by hand), its files are gone: use
    <experiment>_<folder id> instead."""
    import dataiku
    folder_id = dataiku.Folder(folder).get_id()
    found = mlflow.get_experiment_by_name(experiment)
    if found is None or f"//{folder_id}/" in (found.artifact_location or ""):
        return experiment
    return f"{experiment}_{folder_id}"


def new_model_id(cutoff):
    """CFF_<cut-off day>_<training time>, for example CFF_20261031_20261104T1215 (also the saved model version)."""
    return f"CFF_{cutoff:%Y%m%d}_{pd.Timestamp.now():%Y%m%dT%H%M}"


def promotion_rules():
    """The promotion rules: the project variables PROMOTION_RULES names, or their defaults."""
    import dataiku
    variables = dataiku.get_custom_variables()
    rules = {}
    for name, default in PROMOTION_RULES.items():
        value = variables.get(name, default)
        rules[name] = (str(value).strip().lower() in ("true", "1", "yes")) if isinstance(default, bool) else float(value)
    return rules


def saved_model(name=SAVED_MODEL):
    project = _project()
    return project.get_saved_model(next(m["id"] for m in project.list_saved_models() if m["name"] == name))


def forecast_model(forecaster):
    """An MLflow model around a trained forecaster. The nightly forecast unwraps it and calls the forecaster
    (forecast.production_forecaster); called as an MLflow model, it takes daily flows (interface 1) and returns the
    forecast of the 3 windows after the last day it was given, without client details."""
    import mlflow.pyfunc
    from models import forecast

    class ForecastModel(mlflow.pyfunc.PythonModel):
        def __init__(self, forecaster):
            self.forecaster = forecaster

        def predict(self, context, model_input, params=None):
            cutoff = pd.to_datetime(model_input["day"]).max().normalize()
            return forecast(self.forecaster, model_input, None, cutoff)

    return ForecastModel(forecaster)


def _carry_code(forecaster):
    """The models carry their own copy of the library code (pickled by value: cashflow, models, tracking and the
    forecaster's own module, forecasters.py for example), so they keep working when the library changes."""
    import cloudpickle
    import cashflow
    import models
    own = sys.modules.get(type(forecaster).__module__)
    for module in (cashflow, models, sys.modules[__name__], own):
        if module is not None and module.__name__ != "__main__":
            cloudpickle.register_pickle_by_value(module)


def _record(mlflow, params, metrics, backtest):
    """A run's settings, test scores and backtest forecaster."""
    import cloudpickle
    mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
    mlflow.log_metrics({k: float(v) for k, v in metrics.items() if pd.notna(v)})
    with tempfile.TemporaryDirectory() as tmp:
        with open(os.path.join(tmp, "forecaster.pkl"), "wb") as f:
            cloudpickle.dump(backtest, f)
        mlflow.log_artifacts(tmp, artifact_path="backtest")


def log_model(model_id, params, metrics, backtest, final):
    """Record one training run in Experiment Tracking, then deploy its final forecaster as version model_id of the
    saved model cash_forecast (not activated: the promotion does that)."""
    _carry_code(final)
    with _mlflow(FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, EXPERIMENT, FOLDER))
        with mlflow.start_run(run_name=model_id) as run:
            mlflow.set_tag("model_id", model_id)
            _record(mlflow, params, metrics, backtest)
            where = "name" if int(mlflow.__version__.split(".")[0]) >= 3 else "artifact_path"   # MLflow 3 or 2
            mlflow.pyfunc.log_model(python_model=forecast_model(final), **{where: "forecaster"})
            run_id = run.info.run_id
    extension = _project().get_mlflow_extension()
    code_env = _project().get_recipe("train_and_test").get_settings().get_recipe_params()["envSelection"]["envName"]
    extension.set_run_inference_info(run_id=run_id, prediction_type="REGRESSION", code_env_name=code_env)
    # which model of the run: model_name in DSS 15 (MLflow 3 models), model_sub_folder in DSS 14 (the artifact path)
    which = "model_name" if "model_name" in inspect.signature(extension.deploy_run_model).parameters else "model_sub_folder"
    model = saved_model()
    before = (model.get_active_version() or {}).get("id")
    extension.deploy_run_model(run_id, model.id, version_id=model_id, activate=False, skip_expensive_reports=True,
                               **{which: "forecaster"})
    if before and (model.get_active_version() or {}).get("id") != before:
        model.set_active_version(before)       # DSS 14 activates every deployed version: only a promotion may


def log_lab_run(run_name, params, metrics, backtest):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab, folder experiments): its settings, test
    scores and forecaster. Never deployed. Returns the run ID."""
    _carry_code(backtest)
    with _mlflow(LAB_FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, LAB_EXPERIMENT, LAB_FOLDER))
        with mlflow.start_run(run_name=run_name) as run:
            _record(mlflow, params, metrics, backtest)
            return run.info.run_id


def runs():
    """Every training run, newest first: model_id, run_id, trained_at, its settings (params.*) and metrics."""
    with _mlflow(FOLDER) as mlflow:
        # sorted here: DSS 14 cannot sort runs by start time in the search
        found = mlflow.search_runs(experiment_names=[_experiment_name(mlflow, EXPERIMENT, FOLDER)])
    if found.empty:
        return pd.DataFrame(columns=["model_id", "run_id", "trained_at"])
    found = found.sort_values("start_time", ascending=False).reset_index(drop=True)
    return found.rename(columns={"tags.model_id": "model_id", "start_time": "trained_at"})


def latest_model_id():
    """The most recently trained model."""
    found = runs()
    if found.empty:
        raise ValueError("No model trained yet: run the scenario Retrain, test and promote")
    return found.iloc[0]["model_id"]


def production_model_id():
    """The production model: the active version of the saved models, if it was ever promoted (a version Dataiku
    activated on its own when the first one was deployed does not count until the promotion)."""
    model_id = (saved_model().get_active_version() or {}).get("id")
    return model_id if model_id in promoted_model_ids() else ""


def promoted_model_ids():
    """Every model ID promoted so far (read through the API: also works in a recipe that writes promotion_log)."""
    try:
        log = _project().get_dataset("promotion_log")
        columns = [c["name"] for c in log.get_schema()["columns"]]
        model, decision = columns.index("candidate_model_id"), columns.index("decision")
        return {row[model] for row in log.iter_rows() if row[decision] == "PROMOTE"}
    except Exception:            # no promotion yet
        return set()


def load_backtest(model_id):
    """The backtest forecaster of a training run (trained without the last 6 periods, for champion / challenger)."""
    found = runs()
    match = found[found["model_id"] == model_id]
    if match.empty:
        raise ValueError(f"Model {model_id} not found in Experiment Tracking ({EXPERIMENT})")
    with _mlflow(FOLDER) as mlflow:
        path = mlflow.artifacts.download_artifacts(run_id=match.iloc[0]["run_id"], artifact_path="backtest")
    with open(os.path.join(path, "forecaster.pkl"), "rb") as f:
        return pickle.load(f)


def decide(candidate, production, rules=None):
    """Champion / challenger: the decision (PROMOTE or KEEP) and its reason, from the test scores of the candidate and
    of the production model (models.scores), with the promotion rules (promotion_rules: project variables). production
    None: no production model to compare with, the candidate is promoted (the forecast needs one) with its failed
    checks as a warning."""
    rules = rules or promotion_rules()
    low, high = rules["promotion_min_coverage"], rules["promotion_max_coverage"]
    checks = []
    if not low <= candidate["coverage_p10_p90"] <= high:
        checks.append(f"coverage {candidate['coverage_p10_p90']:.1%} outside {low:.0%}-{high:.0%}")
    if rules["promotion_beat_simple_guess"] and candidate["wape_model"] >= candidate["wape_baseline"]:
        checks.append(f"does not beat the simple guess ({candidate['wape_model']:.1%} vs "
                      f"{candidate['wape_baseline']:.1%})")
    if production is None:
        return "PROMOTE", (f"no production model to compare with, EV error {candidate['wape_model']:.2%}, coverage "
                           f"{candidate['coverage_p10_p90']:.1%}" + (f" (warning: {'; '.join(checks)})" if checks else ""))
    if checks:
        return "KEEP", f"candidate {checks[0]}"
    needed = production["wape_model"] * (1 - rules["promotion_min_gain"])
    if candidate["wape_model"] >= needed:
        gain = f" by more than {rules['promotion_min_gain']:.0%}" if rules["promotion_min_gain"] else ""
        return "KEEP", (f"candidate EV error {candidate['wape_model']:.2%} not lower than production "
                        f"{production['wape_model']:.2%}{gain}")
    return "PROMOTE", (f"candidate EV error {candidate['wape_model']:.2%} vs production {production['wape_model']:.2%}, "
                       f"coverage {candidate['coverage_p10_p90']:.1%}")


def apply_promotion():
    """Apply the latest decision in promotion_log: on PROMOTE, activate the candidate's version of the saved model.
    Run by a scenario step after the champion / challenger test."""
    import dataiku
    log = dataiku.Dataset("promotion_log").get_dataframe()
    last = log.sort_values("decided_at").iloc[-1]
    if last["decision"] == "PROMOTE":
        saved_model().set_active_version(last["candidate_model_id"])
        print(f"Promoted {last['candidate_model_id']}: {last['reason']}")
    else:
        print(f"Kept the production model: {last['reason']}")
    return production_model_id()
'''

In [ ]:
FORECAST_CODE = r'''
"""Inference (project library: python/forecast.py): the production model and the nightly forecast. Inference reads
nothing from training except the active version of the saved model cash_forecast (interface 3, cashflow.SAVED_MODEL).
The load test project runs this same file on fake data.

The nightly forecast (forecast_all), Polars from start to end: the groups of clients of daily_flows run in parallel
(`workers` threads: Polars, the Parquet files and CatBoost work outside Python's lock, so threads run side by side;
processes would not be safe here, as Polars and CatBoost keep their own threads), each group on its own from start
to end (forecast_group):
    reading      the files of its last months only, the last input days (365) of its active clients
    preparing    the model's input rows (cashflow.model_rows: client, category, window, day_365 ... day_1)
    model        the model's predict_rows: P10, EV, P90 of each row
    formatting   one row per client (output_rows)
    writing      one Parquet file of the group, straight into the output folder
The output folder holds one subfolder per night, forecast_<cut-off>_<model ID> (kept: the archive), and current.txt,
which points to the newest one only once every group is written: a failed night leaves the previous forecast current.
The SQL table final_forecast is then loaded from it (load_to_sql)."""
import io
import os
import time

import numpy as np
import pandas as pd
import polars as pl

from cashflow import (DAILY_FLOWS, HORIZONS, MIN_HISTORY_DAYS, SAVED_MODEL, TOTAL, Progress, client_groups,
                      day_matrix, input_days, model_rows, peak_memory_mb, read_first_days, read_group, read_manifest,
                      read_parquet, window)

VALUES = ("p10", "ev", "p90")
STEPS = ("reading", "preparing", "model", "formatting", "writing")
OUTPUT = "forecast"                 # the output folder
CURRENT = "/current.txt"            # in it: the subfolder of the current forecast


def _saved_model_id():
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {m["name"]: m["id"] for m in project.list_saved_models()}
    if SAVED_MODEL not in ids:
        raise ValueError(f"Saved model {SAVED_MODEL} not found. Run the setup notebook.")
    return ids[SAVED_MODEL]


def active_model_id():
    """The production model: the active version of the saved model cash_forecast."""
    import dataiku
    model = dataiku.api_client().get_default_project().get_saved_model(_saved_model_id())
    model_id = (model.get_active_version() or {}).get("id")
    if not model_id:
        raise ValueError("No production model yet: run the scenario Retrain, test and promote")
    return model_id


def production_forecaster():
    """The production model (the active version of cash_forecast), and its model ID. The recipe that calls it must
    have the saved model as an input."""
    import dataiku
    model_id = active_model_id()
    predictor = dataiku.Model(_saved_model_id()).get_predictor(model_id)
    return predictor.model.unwrap_python_model().forecaster, model_id


def value_columns():
    """The value columns of the forecast, always the same and in this order: TOTAL first, then each category of
    warehouse.py (CATEGORY_OF_TYPE and OTHER), each for windows 1, 2, 3 (days 1-30, 31-60, 61-90 after the cut-off
    day) and P10, EV, P90: TOTAL_w1_p10, TOTAL_w1_ev, TOTAL_w1_p90, TOTAL_w2_p10, ..., UTILITIES_w3_p90."""
    from warehouse import CATEGORY_OF_TYPE, OTHER
    categories = [TOTAL] + sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER})
    return [f"{c}_w{h}_{v}" for c in categories for h in HORIZONS for v in VALUES]


def output_rows(pred, clients, cutoff):
    """The forecast of a group: ONE row per client of `clients` (client_no, first_day, recent: whether it has a
    posting in its last input days). pred: one row per client, category and window with p10, ev, p90. Columns:
    client_no, cutoff_date (the last day of data), w1_start ... w3_end (the days of each window), status (OK;
    NOT_ENOUGH_HISTORY: less than MIN_HISTORY_DAYS days since its first posting; NO_POSTINGS: none in its last input
    days), history_days, then value_columns() (empty for a category the client does not have, and when not OK)."""
    columns = value_columns()
    wide = (pred.unpivot(index=["client_no", "category", "window"], on=list(VALUES))
            .with_columns(key=pl.format("{}_w{}_{}", "category", "window", "variable"))
            .pivot(on="key", index="client_no", values="value")) if len(pred) else pl.DataFrame(
        schema={"client_no": pl.Utf8})
    unknown = sorted({c.split("_w")[0] for c in wide.columns[1:] if c not in columns})
    if unknown:
        raise ValueError(f"Forecast for categories not in warehouse.py: {', '.join(unknown)}")
    end = cutoff.to_pydatetime().date()
    history = (pl.lit(end) - pl.col("first_day").dt.date()).dt.total_days() + 1
    dates = {"cutoff_date": pl.lit(end)}
    for h in HORIZONS:
        first, last = window(cutoff, h)
        dates[f"w{h}_start"], dates[f"w{h}_end"] = pl.lit(first.date()), pl.lit(last.date())
    out = clients.with_columns(history_days=history.fill_null(0).cast(pl.Int64)).join(wide, on="client_no", how="left")
    status = (pl.when(pl.col("client_no").is_in(wide["client_no"].implode())).then(pl.lit("OK"))
              .when(pl.col("recent") & (pl.col("history_days") < MIN_HISTORY_DAYS)).then(pl.lit("NOT_ENOUGH_HISTORY"))
              .otherwise(pl.lit("NO_POSTINGS")))
    return out.select("client_no", *[v.alias(k) for k, v in dates.items()], status.alias("status"), "history_days",
                      *[pl.col(c).cast(pl.Float64) if c in out.columns else pl.lit(None, pl.Float64).alias(c)
                        for c in columns])


_JOB = {}       # what every group needs, set by forecast_all before the workers start


def forecast_group(group):
    """One group of clients from start to end (see the module's text). Returns its seconds per step and counts."""
    job, times = _JOB, {}
    model, cutoff, days = job["model"], job["cutoff"], job["days"]
    ticked = time.time()
    clients = job["clients"][group]
    since = cutoff - pd.Timedelta(days=days - 1)
    daily = read_group(job["folder"], job["manifest"], group, since).filter(
        pl.col("client_no").is_in(clients["client_no"].implode()))
    clients = clients.with_columns(recent=pl.col("client_no").is_in(daily["client_no"].unique().implode()))
    enough = clients.filter(pl.col("recent") & ((pl.lit(cutoff.to_pydatetime()) - pl.col("first_day")).dt.total_days()
                                                >= MIN_HISTORY_DAYS - 1))
    daily = daily.filter(pl.col("client_no").is_in(enough["client_no"].implode()))
    times["reading"], ticked = time.time() - ticked, time.time()
    rows = model_rows(*day_matrix(daily, cutoff, days))
    times["preparing"], ticked = time.time() - ticked, time.time()
    values = np.asarray(model.predict_rows(rows), dtype=np.float64) if len(rows) else np.zeros((0, 3))
    if values.shape != (len(rows), 3) or not np.isfinite(values).all():
        raise ValueError(f"The model's predict_rows must return one finite P10, EV, P90 per input row: got {values.shape}")
    times["model"], ticked = time.time() - ticked, time.time()
    ev = values[:, 1]
    pred = rows.select("client_no", "category", "window").with_columns(
        p10=np.minimum(values[:, 0], ev).round(2), ev=ev.round(2), p90=np.maximum(values[:, 2], ev).round(2))
    out = output_rows(pred, clients, cutoff).with_columns(**{k: pl.lit(v) for k, v in job["columns"].items()})
    times["formatting"], ticked = time.time() - ticked, time.time()
    buffer = io.BytesIO()
    out.write_parquet(buffer)
    with job["output"].get_writer(f"/{job['version']}/part_{group:05d}.parquet") as writer:
        writer.write(buffer.getvalue())
    times["writing"] = time.time() - ticked
    counts = out["status"].value_counts()
    return {**times, "daily_rows": len(daily), "clients": len(out),
            **dict(zip(counts["status"].to_list(), counts["count"].to_list()))}


def forecast_all(model, model_id, clients, cutoff, workers, name=DAILY_FLOWS, output=OUTPUT, columns=None):
    """The nightly forecast of `clients` (Polars or pandas, client_no: the active clients) with `model`, into the
    folder `output` (see the module's text). columns: values added to every row (forecaster, load_id, ...). Returns
    the seconds of each step (added up over the groups), the wall time, the counts per status, the most memory used
    and the output's subfolder."""
    from concurrent.futures import ThreadPoolExecutor
    import dataiku
    started = time.time()
    folder, manifest = read_manifest(name)
    if manifest is None:
        raise ValueError(f"{name} is empty: run the staging first")
    clients = (clients if isinstance(clients, pl.DataFrame) else pl.from_pandas(clients[["client_no"]])).select(
        pl.col("client_no").cast(pl.Utf8)).unique()
    clients = clients.join(read_first_days(folder, manifest), on="client_no", how="left").with_columns(
        group=client_groups(clients["client_no"], manifest["groups"]))
    by_group = {g: part.drop("group") for (g,), part in clients.group_by(["group"])}
    version = f"forecast_{cutoff:%Y-%m-%d}_{model_id}"
    out = dataiku.Folder(output)
    for path in out.list_paths_in_partition():                  # an earlier try of the same night
        if path.startswith(f"/{version}/"):
            out.delete_path(path)
    workers = max(1, min(int(workers), len(by_group)))
    model.threads = max(1, (os.cpu_count() or 1) // workers)    # CatBoost's threads: the cores shared out
    _JOB.update(model=model, cutoff=cutoff, days=input_days(model), clients=by_group, folder=folder,
                manifest=manifest, output=out, version=version,
                columns={"model_id": model_id, **(columns or {})})
    summary = {k: 0.0 for k in STEPS}
    summary.update(daily_rows=0, groups=len(by_group), workers=workers)
    progress = Progress(f"forecasting {len(clients):,} clients: {len(by_group):,} groups, {workers} at a time",
                        len(clients))
    with ThreadPoolExecutor(workers) as pool:
        for result in pool.map(forecast_group, sorted(by_group)):
            for k, v in result.items():
                if k != "clients":
                    summary[k] = summary.get(k, 0) + v
            progress.add(result["clients"])
    progress.finish()
    with out.get_writer(CURRENT) as writer:
        writer.write(version.encode())
    summary.update(clients=len(clients), seconds=time.time() - started, output=version, peak_memory_mb=peak_memory_mb())
    return summary


def current_output(output=OUTPUT):
    """The output folder, its current subfolder and that subfolder's Parquet files (one per group)."""
    import dataiku
    folder = dataiku.Folder(output)
    paths = folder.list_paths_in_partition()
    if CURRENT not in paths:
        raise ValueError(f"No forecast in the folder {output} yet: run the nightly forecast first")
    with folder.get_download_stream(CURRENT) as stream:
        version = stream.read().decode().strip()
    return folder, version, sorted(p for p in paths if p.startswith(f"/{version}/") and p.endswith(".parquet"))


def load_to_sql(dataset, output=OUTPUT):
    """Load the current forecast of the output folder into the SQL dataset `dataset` (replacing what was there), one
    group's file at a time. Returns the rows loaded and the subfolder."""
    import dataiku
    folder, version, paths = current_output(output)
    sql, writer, rows = dataiku.Dataset(dataset), None, 0
    progress = Progress(f"loading {version} into {dataset} (files)", len(paths))
    try:
        for path in paths:
            frame = read_parquet(folder, path).to_pandas()
            if writer is None:
                sql.write_schema_from_dataframe(frame)
                writer = sql.get_writer()
            writer.write_dataframe(frame)
            rows += len(frame)
            progress.add()
    finally:
        if writer is not None:
            writer.close()
    progress.finish()
    return rows, version
'''

In [ ]:
FORECASTERS_CODE = r'''
"""The data scientist's forecasters (project library: python/forecasters.py). The setup notebook creates this file once
and never overwrites it: it is yours.

Try a forecaster first in the notebook forecast_lab (it never touches production). To put one in production:
    1. add its class here and to FORECASTERS below,
    2. set the project variable forecaster to its name,
    3. run the scenario Retrain, test and promote: champion / challenger promotes it only if it passes the promotion
       rules (project variables promotion_*) on the same last 6 periods of 30 days. To go back, set forecaster to daily_catboost.

A model is a class built on models.DayModel with (details in models.py, and the examples DailyCatBoost and
RecentAverage there):
    name            the value of the project variable forecaster that selects it
    params          its settings (a dict), recorded in Experiment Tracking; input_days: the days it gets (365)
    fit(daily, clients, cutoff)
                    learns from the daily flows (client_no, day, category, money_in, money_out, postings) up to the
                    cut-off day (yesterday), and the clients
    predict_rows(rows)
                    rows: one row per client, category (and TOTAL) and window (1, 2, 3: days 1-30, 31-60, 61-90 after
                    the cut-off day) with day_365 ... day_1 (Polars, cashflow.model_rows); returns P10, EV, P90 per row
                    (numpy, the net cash flow of that window, SAR)
"""
from models import RecentAverage


class RecentAverage6(RecentAverage):
    """Example: recent_average (models.py) with the average of the last 6 periods of 30 days instead of 3."""
    name = "recent_average_6m"
    params = {**RecentAverage.params, "periods": 6}


FORECASTERS = [RecentAverage6]
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["stage_tables"] = r'''
# Staging: the 2 warehouse tables, read-only, turned into the project's tables, all in Dataiku. This is interface 1,
# what the data scientist's models get:
#   clients: the business clients of DIM_CUSTOMER (companies and sole proprietors), standard names and values. Read
#   whole every run (it is small).
#   daily_flows: the postings of FCT_STATEMENT_ENTRY summed per client, day and category (money in, money out):
#   business clients only, no reversals, no transfers between the client's own accounts. Only the NEW days are read
#   from the warehouse: the days after the last day already staged, up to the cut-off day (yesterday). When nothing
#   is staged yet (the first run, or after the scenario Full rebuild), the whole history is read.
#   daily_flows is a folder of Parquet files (read straight from the files, far faster than a dataset): one file per
#   group of clients (a client never changes group) and calendar month (cashflow.write_daily), so the nightly
#   forecast reads only the last months of each group. New days only rewrite the files of their months (usually the
#   current month of each group); the manifest only points to the new files once every one is written, so a failed
#   run leaves daily_flows as it was. When there is nothing new, daily_flows is not touched.
#   staging_log: one row per run (appended), also when the run fails (status FAILED and the error): what was read,
#   and the last day now in daily_flows.
# The warehouse does the heavy work (warehouse.daily_totals_query): it sums the postings into daily totals per client,
# day and category, without reversals and own-account transfers, so only those totals travel, not every posting. A
# big read never sits in memory whole: the totals read and the rows already staged are spread over files on disk by
# group of clients and written one group at a time.
import json
import os
import tempfile
import time

import dataiku
import pandas as pd

from cashflow import (CLIENTS_PER_GROUP, DAILY_COLUMNS, Progress, Spread, append_row, client_groups, clients_limit,
                      cutoff_date, history_start, read_manifest, waiting, write_daily)
import warehouse
from warehouse import TABLES, daily_totals_query, one_row_per_client, standard_daily, table_query, to_standard

CHUNK = 200_000
STEP = 10_000                       # rows: how often the reading progress is counted (CHUNK is a multiple of it)
started = time.time()
cutoff = cutoff_date()
end = cutoff + pd.Timedelta(days=1)                # postings up to the cut-off day (included)


def read(query, progress, convert):
    """Runs a read-only query through the warehouse connection and yields its rows in chunks (DataFrames, converted
    by convert). progress counts the rows as they arrive. The query is printed first, and until its first row
    arrives a line says every minute that the warehouse is still running it."""
    print(f"PROGRESS SQL: {query}", flush=True)
    started, first = time.time(), waiting(progress.what)
    try:
        reader = dataiku.SQLExecutor2(dataset=dataiku.Dataset(TABLES["clients"][0])).query_to_iter(query)
        names, rows = [c["name"] for c in reader.get_schema()], []
        for row in reader.iter_tuples():
            if not first.is_set():
                first.set()
                print(f"PROGRESS {progress.what}: first rows after {(time.time() - started) / 60:.1f} min",
                      flush=True)
            rows.append(row)
            if len(rows) % STEP == 0:
                progress.add(STEP)
            if len(rows) == CHUNK:
                yield convert(pd.DataFrame(rows, columns=names))
                rows = []
        progress.add(len(rows) % STEP)
        if rows:
            yield convert(pd.DataFrame(rows, columns=names))
    finally:
        first.set()


def table_name(table):
    return sql_name(dataiku.Dataset(TABLES[table][0]))


def sql_name(dataset):
    """The table's full quoted name. The project reads the warehouse through SQL only: anything else stops here."""
    try:
        name = dataset.get_location_info()["info"].get("quotedResolvedTableName")
    except Exception:
        name = None
    if not name:
        raise ValueError(f"{dataset.name} is not a database table: the project reads the warehouse through SQL only. "
                         "Run the setup notebook again (it links the warehouse tables)")
    return name


LOG = {"run_at": "string", "status": "string", "error": "string", "mode": "string", "cutoff_date": "string",
       "read_from": "string", "read_to": "string", "postings_read": "bigint", "postings_kept": "bigint",
       "daily_rows_added": "bigint", "daily_rows_total": "bigint", "last_day": "string", "postings_per_month": "string",
       "clients": "bigint", "seconds": "double"}
row = {"run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"), "cutoff_date": f"{cutoff:%Y-%m-%d}",
       "read_to": f"{cutoff:%Y-%m-%d}"}


def stage():
    """Reads the new days and adds them to daily_flows; fills row (the staging_log row) as it goes."""
    # what is already staged: staging_log (this recipe adds to it, so it reads it outside the flow's inputs). Failed
    # runs changed nothing.
    try:
        log = dataiku.Dataset("staging_log", ignore_flow=True).get_dataframe()
    except Exception:
        log = pd.DataFrame(columns=list(LOG))
    log = log[log["status"] == "SUCCESS"] if len(log) else log
    last_day = pd.to_datetime(log["last_day"]).max() if len(log) else None
    staged_rows = int(log.sort_values("run_at")["daily_rows_total"].iloc[-1]) if len(log) else 0
    _, manifest = read_manifest(ignore_flow=True)
    if last_day is None and manifest is not None:
        raise ValueError("staging_log is empty but daily_flows is not: run the scenario Full rebuild")
    if manifest is not None and manifest["rows"] != staged_rows:
        raise ValueError(f"daily_flows has {manifest['rows']:,} rows but staging_log says {staged_rows:,}: run the "
                         "scenario Full rebuild of the daily flows")
    since = None if last_day is None else last_day + pd.Timedelta(days=1)
    mode = "FULL" if since is None else "NOTHING_NEW" if since >= end else "NEW_DAYS"
    if mode == "FULL":
        since = history_start(cutoff)              # project variable history_months (None: the whole history)
    row.update(mode=mode, read_from=f"{since:%Y-%m-%d}" if since is not None and mode != "NOTHING_NEW" else "")

    limit = clients_limit()                        # project variable clients_limit (None: all the business clients)
    print(f"PROGRESS staging ({mode})" + (f", the first {limit:,} business clients only" if limit else ""), flush=True)
    progress = Progress("reading the clients from the warehouse (rows)")
    clients = one_row_per_client(pd.concat(
        [one_row_per_client(chunk) for chunk in read(table_query("clients", table_name("clients"), limit=limit), progress,
                                                   lambda df: to_standard(df, "clients"))], ignore_index=True))
    progress.finish()
    print(f"PROGRESS {len(clients):,} business clients", flush=True)
    if progress.done > len(clients):               # DIM_CUSTOMER keeps history: several rows per client
        print(f"WARNING: DIM_CUSTOMER sent {progress.done:,} rows for {len(clients):,} business clients (several "
              "rows per client). " + ("Its condition CLIENTS_CURRENT_ROW does not keep one row per client: check it"
                                      if warehouse.CLIENTS_CURRENT_ROW.strip() else
                                      "Fill in CLIENTS_CURRENT_ROW in warehouse.py (the notebook's WAREHOUSE_CODE "
                                      "cell) to read only each client's current row")
              + ". Until then, the last row received is kept, which may not be the newest.", flush=True)
    if clients.empty:
        raise ValueError("No business client in DIM_CUSTOMER: check CLIENTS_BUSINESS and CLIENTS_CURRENT_ROW in "
                         "warehouse.py (the notebook's WAREHOUSE_CODE cell)")
    numbers = set(clients["client_no"])
    dataiku.Dataset("clients").write_with_schema(clients)
    row["clients"] = len(clients)

    read_rows = kept = added = total = 0          # postings in the daily totals read, and kept
    new = {}                                       # postings per month read now
    n_groups = manifest["groups"] if mode == "NEW_DAYS" else max(1, -(-len(clients) // CLIENTS_PER_GROUP))
    with tempfile.TemporaryDirectory() as spread:
        if mode != "NOTHING_NEW":
            groups = Spread(os.path.join(spread, "groups"), min(n_groups, 512))     # whole groups of clients each
            query = daily_totals_query(table_name("transactions"), table_name("clients"), since, end, limit)
            progress = Progress("reading the daily totals from the warehouse (rows: client x day x category)")
            for chunk in read(query, progress, standard_daily):
                read_rows += int(chunk["postings"].sum())
                chunk = chunk[chunk["client_no"].isin(numbers)]
                kept += int(chunk["postings"].sum())
                if len(chunk):
                    last_day = max(chunk["day"].max(), last_day or pd.Timestamp.min)
                    for month, n in chunk.groupby(chunk["day"].dt.strftime("%Y-%m"))["postings"].sum().items():
                        new[month] = new.get(month, 0) + int(n)
                    groups.add(chunk[DAILY_COLUMNS], client_groups(chunk["client_no"], n_groups).to_pandas())
            progress.finish()
            row.update(postings_read=read_rows, postings_kept=kept)

        # the postings must reach the cut-off day (yesterday): otherwise the warehouse load looks late. Checked before
        # anything is written, so a failed run changes nothing and the next run reads the same days again.
        if last_day is None or last_day < cutoff:
            raise ValueError(f"FCT_STATEMENT_ENTRY has no posting of business clients after "
                             f"{'(none)' if last_day is None else f'{last_day:%Y-%m-%d}'}: the warehouse load of "
                             f"{cutoff:%Y-%m-%d} (the cut-off day) looks late")

        if mode == "NOTHING_NEW":
            total = staged_rows                    # daily_flows is not touched
        else:
            # the new rows go to the files of their group of clients and month (added to the rows already there)
            progress = Progress(f"writing daily_flows ({n_groups:,} groups of clients, files by month)", groups.n)
            total = write_daily((counted(part, progress) for part in groups.parts()), n_groups,
                                base=manifest if mode == "NEW_DAYS" else None)["rows"]
            progress.finish()
            added = total - staged_rows
    row.update(postings_read=read_rows, postings_kept=kept, daily_rows_added=added, daily_rows_total=total,
               last_day=f"{last_day:%Y-%m-%d}", postings_per_month=json.dumps(dict(sorted(new.items()))))


def counted(part, progress):
    progress.add()
    return part


def write_log(status, error=""):
    row.update(status=status, error=error[:1000], seconds=round(time.time() - started, 1))
    append_row("staging_log", LOG, row)


try:
    stage()
except Exception as e:
    write_log("FAILED", f"{type(e).__name__}: {e}")
    raise
write_log("SUCCESS")
print(f"Staging done ({row['mode']}), last day staged {row['last_day']}:\n"
      f"- clients: {row['clients']:,} business clients\n"
      f"- postings read" + (f" since {row['read_from']}" if row["read_from"] else "") + f": {row['postings_read']:,} "
      "(a posting = one debit or credit line of FCT_STATEMENT_ENTRY, reversals and own-account transfers left out)"
      + (f", {row['postings_kept']:,} of them for the clients above" if row["postings_kept"] != row["postings_read"]
         else "") + "\n"
      f"- daily_flows: {row['daily_rows_added']:,} rows added, {row['daily_rows_total']:,} in all (a row = one "
      "client, one day, one category: money in, money out and how many postings, the postings summed)")
'''

In [ ]:
RECIPES["train_and_test"] = r'''
# Train and test one model, with its own model ID, tracked in Experiment Tracking with its forecaster, settings and
# scores. The forecaster is the one the project variable forecaster names (default daily_catboost, in models.py; the
# data scientist's own are in forecasters.py). It learns from the daily flows and the clients (interface 1).
#   backtest: trained without the last 6 periods of 30 days, then tested on them (models.backtest, test_results)
#   final: trained on everything up to the cut-off, deployed as version <model ID> of the saved model cash_forecast
#   (not active: the promotion activates it)
import time

import dataiku
import pandas as pd

from cashflow import MIN_TRAINING_MONTHS, cutoff_date, read_daily, read_dataset, until
from models import backtest, new_forecaster, run_metrics, test_results
from tracking import log_model, new_model_id

started = time.time()


def step(what):
    """A progress line in the job's log (the setup notebook shows it while it waits)."""
    print(f"PROGRESS training: {what} ({(time.time() - started) / 60:.1f} min so far)", flush=True)


cutoff = cutoff_date()
step("reading daily_flows and clients")
daily, clients = until(read_daily(), cutoff), read_dataset("clients")
first_day = pd.to_datetime(daily["day"]).min() if len(daily) else cutoff
if first_day > cutoff - pd.DateOffset(months=MIN_TRAINING_MONTHS) + pd.Timedelta(days=1):
    raise ValueError(f"Not enough history to train and test a model: daily_flows starts on {first_day:%Y-%m-%d}, and "
                     f"a model needs at least {MIN_TRAINING_MONTHS} months up to the cut-off day ({cutoff:%Y-%m-%d}). "
                     "Run the setup again with more HISTORY_MONTHS, then the scenario Full rebuild")
backtested, final = new_forecaster(), new_forecaster()
step(f"1/2 test: training without the last periods, then testing on them ({len(daily):,} daily rows)")
tested = backtest(backtested, daily, clients, cutoff)
step("2/2 training the final model on everything up to the cut-off")
final.fit(daily, clients, cutoff)
step("saving and deploying the model")
model_id = new_model_id(cutoff)

results = test_results(tested)
results.insert(0, "model_id", model_id)
metrics = run_metrics(results)
params = {"forecaster": final.name, **getattr(final, "params", {}), "cutoff_date": f"{cutoff:%Y-%m-%d}",
          "load_id": dataiku.get_custom_variables().get("load_id", ""), "daily_rows": len(daily),
          "clients": daily["client_no"].nunique(), "test_rows": len(tested)}
log_model(model_id, params, metrics, backtested, final)
print(f"Model {model_id} ({final.name}): EV error {metrics['wape']:.1%} (simple guess {metrics['wape_baseline']:.1%}), "
      f"coverage {metrics['coverage_p10_p90']:.1%}")

dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["champion_challenger"] = r'''
# Champion / challenger: the latest trained model (candidate) against the production model, both tested on the
# same last 6 periods of 30 days with their backtest forecasters (trained without those periods). The rules are project variables
# (tracking.PROMOTION_RULES, applied by tracking.decide): promoted if its P10-P90 coverage is in the range
# (default 70%-90%), it beats the simple guess (the same as the month before) and its EV error is lower than the
# production model's (by more than promotion_min_gain; a tie keeps the production model). If there is no production
# model to compare with (none yet, or its tracked files were deleted), the candidate is promoted (the forecast needs
# one), with its failed checks as a warning. The production model is the active version of the saved model
# cash_forecast. The decision is added to promotion_log; the next scenario step applies it (activates the
# candidate's version).
import dataiku
import pandas as pd

from cashflow import cutoff_date, read_daily, read_dataset, until
from models import scores, test_forecasts
from tracking import decide, latest_model_id, load_backtest, production_model_id, promotion_rules, runs

cutoff = cutoff_date()
daily, clients = until(read_daily(), cutoff), read_dataset("clients")
candidate = latest_model_id()
production = production_model_id()
tracked = runs()
if production and production not in set(tracked["model_id"]):     # its tracked files were deleted: nothing to test
    print(f"Production model {production} is not in Experiment Tracking: the candidate is judged on its own")
    production = ""
tested = {m: scores(test_forecasts(load_backtest(m), daily, clients, cutoff)) for m in {candidate, production} if m}
c = tested[candidate]
p = tested.get(production) if production and production != candidate else None
if production == candidate:
    decision, reason = "KEEP", "the candidate is already the production model"
else:
    decision, reason = decide(c, p, promotion_rules())
print(decision, "-", reason)

forecaster = dict(zip(tracked["model_id"], tracked["params.forecaster"])) if "params.forecaster" in tracked else {}
dataiku.Dataset("promotion_log").write_with_schema(pd.DataFrame([{
    "decided_at": pd.Timestamp.now().floor("s"), "cutoff_date": dataiku.get_custom_variables()["cutoff_date"],
    "candidate_model_id": candidate, "candidate_forecaster": forecaster.get(candidate),
    "production_model_id": production or None, "production_forecaster": forecaster.get(production),
    "candidate_wape": c["wape_model"], "production_wape": p["wape_model"] if p is not None else None,
    "baseline_wape": c["wape_baseline"], "candidate_coverage": c["coverage_p10_p90"],
    "production_coverage": p["coverage_p10_p90"] if p is not None else None,
    "decision": decision, "reason": reason, "rules": str(promotion_rules())}]))
'''

In [ ]:
RECIPES["nightly_forecast"] = r'''
# Nightly forecast: the production model (the active version of the saved model cash_forecast) forecasts, for each
# active business client, the net cash flow (money in minus money out) of each of the next 3 windows of 30 days after
# the cut-off day (yesterday): days 1-30, 31-60 and 61-90, per category and in TOTAL, as P10, EV and P90 in SAR, from
# the client's daily flows of the last 365 days (interface 1). Inference reads nothing else from training.
# forecast.forecast_all: the groups of clients of daily_flows in parallel (project variable inference_workers), each
# read, prepared, forecast and written on its own into the folder forecast: one row per client (status OK,
# NOT_ENOUGH_HISTORY or NO_POSTINGS, the values as columns TOTAL_w1_p10, ...), one subfolder per night,
# forecast_<cut-off>_<model ID>, which becomes the current one once every group is written. The next step of the
# Nightly batch loads it into the SQL table final_forecast (recipe load_forecast_sql).
# inference_log: one row per run (appended), also when the run fails (status FAILED and the error): which model and
# load, how many active clients got a forecast or not and why, the seconds of each step (added up over the workers)
# and in all, the most memory used (all the workers together).
import time

import dataiku
import pandas as pd

from cashflow import append_row, cutoff_date, input_days, read_dataset, whole_number
from forecast import STEPS, forecast_all, production_forecaster

LOG = {"run_at": "string", "status": "string", "error": "string", "load_id": "string", "cutoff_date": "string",
       "model_id": "string", "forecaster": "string", "input_days": "bigint", "workers": "bigint", "groups": "bigint",
       "active_clients": "bigint", "clients_ok": "bigint", "clients_not_enough_history": "bigint",
       "clients_no_postings": "bigint", "daily_rows_read": "bigint",
       **{f"seconds_{k}": "double" for k in STEPS}, "seconds_total": "double", "peak_memory_mb": "bigint",
       "output": "string"}
started = time.time()
variables = dataiku.get_custom_variables()
log = {"run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"), "load_id": variables.get("load_id", ""),
       "cutoff_date": variables.get("cutoff_date", "")}


def run():
    model, model_id = production_forecaster()
    if not hasattr(model, "predict_rows"):
        raise ValueError(f"The production model {model_id} has no predict_rows (models.py): run the scenario Retrain, "
                         "test and promote")
    clients = read_dataset("clients")
    clients = clients[clients["status"] == "ACTIVE"]
    if clients.empty:
        raise ValueError("No active client in clients: check the staging and STS_FLG in warehouse.py")
    name = getattr(model, "name", type(model).__name__)
    log.update(model_id=model_id, forecaster=name, input_days=input_days(model), active_clients=len(clients))
    workers = whole_number("inference_workers", variables.get("inference_workers")) or 4
    done = forecast_all(model, model_id, clients, cutoff_date(), workers,
                        columns={"forecaster": name, "load_id": variables.get("load_id", "")})
    log.update(workers=done["workers"], groups=done["groups"], daily_rows_read=done["daily_rows"],
               clients_ok=done.get("OK", 0), clients_not_enough_history=done.get("NOT_ENOUGH_HISTORY", 0),
               clients_no_postings=done.get("NO_POSTINGS", 0), peak_memory_mb=done["peak_memory_mb"],
               output=done["output"], **{f"seconds_{k}": round(done[k], 1) for k in STEPS})


def write_log(status, error=""):
    log.update(status=status, error=error[:1000], seconds_total=round(time.time() - started, 1))
    append_row("inference_log", LOG, log)


try:
    run()
except Exception as e:
    write_log("FAILED", f"{type(e).__name__}: {e}")
    raise
write_log("SUCCESS")
print(f"Forecast with {log['model_id']} ({log['forecaster']}): {log['active_clients']:,} active clients, "
      f"{log['groups']} groups, {log['workers']} workers: {log['clients_ok']:,} OK, {log['clients_not_enough_history']:,} "
      f"without enough history, {log['clients_no_postings']:,} without postings; {log['seconds_total']:.0f} s; "
      f"folder forecast, {log['output']}")
'''

In [ ]:
RECIPES["load_forecast_sql"] = r'''
# The Nightly batch's last step: the current forecast (folder forecast, written by nightly_forecast) loaded into the
# SQL table final_forecast, replacing the previous one: one row per active client. If it fails, the forecast itself is
# complete in the folder: run this step again alone (build final_forecast).
from forecast import load_to_sql

rows, version = load_to_sql("final_forecast")
print(f"final_forecast: {rows:,} rows loaded from {version}")
'''

## 3. Build the flow

Uses the simulated warehouse tables, or links the warehouse's (read-only), and creates the Python recipes and their
datasets (CSV files, and the SQL table `final_forecast`), the folders `daily_flows` (Parquet files), `experiments`
(tracked models) and `forecast` (Parquet, one subfolder per night), the saved model `cash_forecast`, and the project
variables `forecaster` (`daily_catboost`), `inference_workers` (4) and the promotion rules (`promotion_*`) if they are
not set yet, in 4 zones.
Training (folder `experiments`) and the promotion log are only rebuilt when asked by name, so the nightly batch never
retrains by accident.

In [ ]:
import json
import re
import sys
import time
import types

import pandas as pd

# Safe to run again in the same project: what already exists is kept (datasets and their data, folders, the saved
# model and its versions, zones, the project variables, forecasters.py), and the code of the Python recipes and of the
# rest of the library is updated.
flow = project.get_flow()


def load_module(name, code):
    """A library file as a module of this notebook: the code it writes to the project library."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(code, module.__dict__)
    return module


cashflow = load_module("cashflow", LIBRARY_CODE)
warehouse = load_module("warehouse", WAREHOUSE_CODE)
tracking = load_module("tracking", TRACKING_CODE)
RAW = [warehouse_table for warehouse_table, _ in warehouse.TABLES.values()]

# the input, checked before anything is built: the 2 warehouse tables, read-only through EDW_CONNECTION (SQL only)
try:
    database = client.get_connection(EDW_CONNECTION).get_info().get_type()      # Oracle, PostgreSQL, ...
except Exception as e:
    raise RuntimeError(f"Cannot use the connection {EDW_CONNECTION!r} ({e}). Check EDW_CONNECTION, or ask a Dataiku "
                       "admin for a read-only database connection to the warehouse, usable by your group.") from None
print(f"Input: {database} connection {EDW_CONNECTION}, read-only")
for table, (name, _) in warehouse.TABLES.items():
    columns = warehouse.read_columns(table)
    query = warehouse.table_query(table, f'"{EDW_SCHEMA}"."{name}"', end=pd.Timestamp.now())
    try:
        dataiku.SQLExecutor2(connection=EDW_CONNECTION).query_to_df(f"SELECT * FROM ({query}) q WHERE 1 = 0")
    except Exception as e:
        found = re.search(r"(ORA-\d+[^\\\n']*|ERROR: [^\\\n']*)", str(e))    # the database's own message
        reason = found.group(1).strip() if found else str(e).splitlines()[0][:200]
        raise RuntimeError(f"Cannot read {EDW_SCHEMA}.{name} through {EDW_CONNECTION} ({reason}). Check EDW_SCHEMA, "
                           f"that the table exists with the columns {', '.join(columns)}, and that the connection's "
                           "user may read it.") from None
    print(f"   {EDW_SCHEMA}.{name}: can be read ({len(columns)} columns used)")
try:
    client.get_connection(OUTPUT_SQL_CONNECTION).get_info()
except Exception as e:
    raise RuntimeError(f"Cannot use the connection {OUTPUT_SQL_CONNECTION!r} ({e}). Check OUTPUT_SQL_CONNECTION: a "
                       "database connection your group may write into (never the warehouse).") from None
print(f"Output: the table final_forecast on {OUTPUT_SQL_CONNECTION}, datasets on {OUTPUT_CONNECTION}, folders on "
      f"{OUTPUT_FOLDER_CONNECTION}")
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}
SQL = ["final_forecast"]              # a table on OUTPUT_SQL_CONNECTION; every other dataset is a CSV file

# Python recipes: (inputs, outputs, flow zone). daily_flows (Parquet files, read far faster than a dataset),
# experiments and forecast (Parquet, one subfolder per night) are folders; cash_forecast is the saved model.
# Each zone has an owner, and the zones only meet through three interfaces:
#   1. the data: daily_flows and clients (Staging) -> Training and Inference (and the data scientist's lab notebook)
#   2. the forecaster (models.py, forecasters.py): the project variable forecaster names the one Retrain trains. The
#      data scientist tries forecasters first in the notebook forecast_lab, outside the flow, without touching
#      production.
#   3. the saved model's active version: training -> inference. Inference reads nothing else from training.
# The load test is a project of its own (build_load_test_project.ipynb): nothing of it is in this project.
DATA = ["daily_flows", "clients"]
FLOW = {
    "stage_tables": (RAW, warehouse.STAGED, "Staging"),
    "train_and_test": (DATA, ["experiments", "test_results"], "Training"),
    "champion_challenger": (DATA + ["experiments"], ["promotion_log"], "Training"),
    "nightly_forecast": (DATA + [cashflow.SAVED_MODEL], ["forecast", "inference_log"], "Inference"),
    "load_forecast_sql": (["forecast"], ["final_forecast"], "Inference"),
}
FOLDERS = ["daily_flows", "experiments", "forecast"]
# logs: each run adds rows
APPENDED = ["staging_log", "promotion_log", "inference_log"]
BUILT_BY_NAME = ["experiments", "promotion_log"]   # only built when asked by name

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

# the project library: shared (cashflow.py, warehouse.py), training and lab (models.py, tracking.py), inference
# (forecast.py). forecasters.py is the data scientist's: written only when it is created, never overwritten.
library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE), ("models.py", MODELS_CODE),
                        ("tracking.py", TRACKING_CODE), ("forecast.py", FORECAST_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)
if not python_folder.get_file("forecasters.py"):
    python_folder.add_file("forecasters.py").write(FORECASTERS_CODE)
    print("forecasters.py created (the data scientist's forecasters)")

# project variables, kept if they are already set: forecaster (the one Retrain trains), inference_workers (how many
# groups of clients the nightly forecast runs at a time)
variables = project.get_variables()
variables["standard"].pop("inference_clients_per_chunk", None)   # the older setting
defaults = {"forecaster": "daily_catboost", "inference_workers": 4, **tracking.PROMOTION_RULES}
missing = {k: v for k, v in defaults.items() if variables["standard"].get(k) in (None, "")}
if variables["standard"].get("forecaster") == "quantile_boosting":          # the earlier placeholder, removed
    missing["forecaster"] = "daily_catboost"
    print("Project variable forecaster: quantile_boosting (removed) replaced by daily_catboost")
if missing:
    variables["standard"].update(missing)
    project.set_variables(variables)
# history_months, clients_limit: always the setup's settings (what staging reads: the first time and Full rebuild)
months = cashflow.whole_number("HISTORY_MONTHS", HISTORY_MONTHS)
limit = cashflow.whole_number("CLIENTS_LIMIT", CLIENTS_LIMIT)
variables["standard"].pop("history_years", None)                 # the older setting
variables["standard"].update(history_months="" if months is None else months, clients_limit="" if limit is None else limit)
project.set_variables(variables)
print("History read the first time (and by Full rebuild):",
      "the whole history" if months is None else f"the last {months} months up to the cut-off day",
      "| clients:", "all the business clients" if limit is None else f"the first {limit:,} business clients")
if months is not None and months < cashflow.MIN_TRAINING_MONTHS:
    print(f"WARNING: with {months} months of history, the first run only stages the data: training, promotion and "
          f"the forecast need at least {cashflow.MIN_TRAINING_MONTHS} months")
print("Models: Retrain trains {forecaster}; the nightly forecast runs {inference_workers} groups of clients at a "
      "time".format(**variables["standard"]))
print("Promotion rules:", {k: variables["standard"][k] for k in tracking.PROMOTION_RULES})


# what the project already has (read once, kept up to date below)
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}



def new_dataset(name):
    """Managed dataset of the project: a table on OUTPUT_SQL_CONNECTION (SQL), else a CSV file (comma-separated, with a
    header row) on OUTPUT_CONNECTION. Kept if it exists."""
    if name in datasets:
        return name
    datasets.add(name)
    dataset = project.new_managed_dataset(name).with_store_into(
        OUTPUT_SQL_CONNECTION if name in SQL else OUTPUT_CONNECTION).create()
    settings = dataset.get_settings()
    if name not in SQL:
        settings.get_raw()["formatType"] = "csv"
        settings.get_raw()["formatParams"].update(CSV)
    if name in BUILT_BY_NAME:
        settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
    settings.save()
    return name


# the 2 warehouse tables: datasets linked read-only to the tables of EDW_SCHEMA through EDW_CONNECTION (nothing is
# created or written there). One already in the project is kept only if it is that table (a run of the setup again);
# anything else (fake data, CSV, an old project's table) stops the setup before it is used.
for name in RAW:
    if name in datasets:
        raw = project.get_dataset(name).get_settings().get_raw()
        params = raw.get("params", {})
        if (params.get("connection"), params.get("schema"), params.get("table")) != (EDW_CONNECTION, EDW_SCHEMA, name):
            raise RuntimeError(f"The project already has a dataset {name} that is not the table {EDW_SCHEMA}.{name} "
                               f"of {EDW_CONNECTION} (it is {raw.get('type')} on {params.get('connection')}): fake or "
                               f"old data. Delete it (Flow > {name} > Delete), or use a new empty project, then run "
                               "the setup again.")
        print(f"{name}: already linked to {EDW_SCHEMA}.{name} of {EDW_CONNECTION}")
        continue
    datasets.add(name)
    dataset = project.create_sql_table_dataset(name, database, EDW_CONNECTION, name, EDW_SCHEMA)
    dataset.autodetect_settings().save()           # the table's columns, read from the database
    if not dataset.get_settings().get_raw().get("schema", {}).get("columns"):
        # staging reads with its own SELECT, so it still works; only Dataiku's view of the table would fail
        print(f"WARNING: Dataiku could not read the columns of {EDW_SCHEMA}.{name}: opening the dataset in Dataiku "
              "will fail. Fix: the dataset's Settings > Test table & get schema > Save.")
    print(f"{name}: linked to {EDW_SCHEMA}.{name} of {EDW_CONNECTION}")
override = variables["standard"].get("cutoff_override")
if override:
    print(f"WARNING: the project variable cutoff_override is {override}: the forecast uses that day instead of "
          "yesterday. Delete it (Project > Variables) unless you mean it.")

existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {}
for name in FOLDERS:
    if name in existing_folders:
        folders[name] = project.get_managed_folder(existing_folders[name])
        continue
    folders[name] = project.create_managed_folder(name, connection_name=OUTPUT_FOLDER_CONNECTION)
    if name in BUILT_BY_NAME:
        folder_settings = folders[name].get_settings()
        folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
        folder_settings.save()
existing_models = {m["name"]: m["id"] for m in project.list_saved_models()}
saved_model = (project.get_saved_model(existing_models[cashflow.SAVED_MODEL]) if cashflow.SAVED_MODEL in existing_models
               else project.create_mlflow_pyfunc_model(cashflow.SAVED_MODEL, "REGRESSION"))
ref = lambda name: (folders[name].id if name in folders else saved_model.id if name == cashflow.SAVED_MODEL
                    else name)

for name, (inputs, outputs, _) in FLOW.items():
    for item in inputs + outputs:
        if item not in folders and item != cashflow.SAVED_MODEL:
            new_dataset(item)
    if name in recipes:                # already there: its inputs and outputs changed? then made again
        settings = project.get_recipe(name).get_settings()
        if (set(settings.get_flat_input_refs()) != set(map(ref, inputs))
                or set(settings.get_flat_output_refs()) != set(map(ref, outputs))):
            project.get_recipe(name).delete()
            recipes.discard(name)
            print("recipe", name, "has new inputs or outputs: made again")
    if name in recipes:                # already there: only its code and code env are updated
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o in APPENDED)
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

ZONES = {"Raw data": RAW, "Staging": list(warehouse.STAGED), "Training": [], "Inference": []}
for _, outputs, zone_name in FLOW.values():
    ZONES[zone_name] += [o for o in outputs if o not in ZONES[zone_name]]
# the raw data goes in Dataiku's Default zone, renamed (so there is no empty zone). It exists once another zone does.
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {z: existing_zones.get(z) or flow.create_zone(z) for z in ZONES if z != "Raw data"}
zones["Raw data"] = existing_zones.get("Raw data") or flow.get_default_zone()
if zones["Raw data"].name != "Raw data":
    zone_settings = zones["Raw data"].get_settings()
    zone_settings.name = "Raw data"
    zone_settings.save()


def safe_add(zone, item):
    """Put an item in a zone (already there: nothing to do)."""
    try:
        zone.add_item(item)
    except Exception as e:
        if "already" not in str(e).lower():
            raise


for zone_name, items in ZONES.items():
    for item in items:
        safe_add(zones[zone_name], folders[item] if item in folders else project.get_dataset(item))
safe_add(zones["Inference"], saved_model)    # next to the recipe that uses it
print("Flow ready:", ", ".join(zones))

## 4. First run

Builds the daily flows (the whole history, the first time), trains, tests and deploys a model, runs champion / challenger (with no production model yet,
the first model is promoted, with a warning in promotion_log if its checks fail), then runs the nightly batch:
forecast, log, SQL load.

In [ ]:
import time


def build(step, items, mode):
    """Runs a Dataiku job and waits for it, showing its PROGRESS lines here (also in the job's log) every 30 s."""
    print(f"\n{step} ...", flush=True)
    builder = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            builder.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            builder.with_output(item)
    job, shown, began = builder.start(), 0, time.time()
    while True:
        state = job.get_status().get("baseStatus", {}).get("state", "")
        try:
            lines = [l[l.index("PROGRESS ") + 9:] for l in job.get_log().splitlines() if "PROGRESS " in l]
        except Exception:
            lines = []
        for line in lines[shown:]:
            print("   ", line, flush=True)
        shown = max(shown, len(lines))
        if state in ("DONE", "FAILED", "ABORTED"):
            break
        time.sleep(30 if time.time() - began > 60 else 5)
    if state != "DONE":
        raise RuntimeError(f"{step}: the job {state}. Open it in Jobs (id {job.id}) to read its log.")
    print(f"{step}: done in {(time.time() - began) / 60:.1f} min", flush=True)



started = time.time()
warehouse.set_cutoff()                                          # the first 2 steps of every scenario
warehouse.check_warehouse_loaded()
build("1/5 Staging: read the warehouse (clients and daily flows)", ["daily_flows"], "RECURSIVE_FORCED_BUILD")
staged_only = months is not None and months < cashflow.MIN_TRAINING_MONTHS
if staged_only:                                                # too little history to train: stop after staging
    print(f"\nStaging done in {(time.time() - started) / 60:.1f} minutes. Steps 2-5 (training, promotion, first "
          f"forecast) are skipped: they need at least {cashflow.MIN_TRAINING_MONTHS} months of history, and "
          f"HISTORY_MONTHS is {months}.")
else:
    build("2/5 Train, test, track and deploy the first model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD")
    build("3/5 Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD")
    production = tracking.apply_promotion()                    # activates its version of cash_forecast
    build("4/5 Forecast every active client and log the run", ["inference_log"], "RECURSIVE_FORCED_BUILD")
    build("5/5 Load the forecast into the SQL table final_forecast", ["final_forecast"], "NON_RECURSIVE_FORCED_BUILD")
    print(f"First run done in {(time.time() - started) / 60:.1f} minutes: production model {production}")

## 5. Scenarios

- **Nightly batch**: set the cut-off, add the new days to the daily flows, forecast with the production model and
  log the run, then load the forecast into the SQL table `final_forecast`. It never trains.
- **Retrain, test and promote**: set the cut-off, add the new days to the daily flows, train, test and deploy a new model with
  the forecaster the project variable `forecaster` names, champion / challenger with the promotion rules, activate
  it if it passes. It does not forecast: the next Nightly batch uses the new active version.

- **Full rebuild of the daily flows** (run by hand, no trigger): empty `daily_flows` and `staging_log`, then read the
  whole history again. For example after a reversal cancels a posting from a day already staged.

Their triggers (daily at 06:00 and monthly on the 5th, Riyadh time) are created switched off; switch them on
in production.

In [ ]:
# Production runs: two independent scenarios. Nightly batch (inference) forecasts with the active version of the saved
# models; Retrain, test and promote (training) adds a version and activates it if it is better, and never forecasts.
# Each starts by setting the cut-off day (yesterday) and checking the warehouse load of that day is finished (Python
# steps; the check is a mock to replace with the bank's); the staging recipe adds the new days to daily_flows and
# stops if the postings do not reach the cut-off day. A third one, Full rebuild of the daily flows, has no trigger:
# run it by hand to read the whole history again. The tracked models (folder experiments) and the promotion log are only rebuilt when built by name, so the
# nightly batch never retrains by accident.
def step(n, label, what, mode=None):
    common = {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
              "runConditionStatuses": ["SUCCESS", "WARNING"]}
    if isinstance(what, dict):          # variables for the next steps, edited on the scenario's page
        return {**common, "type": "define_vars", "params": {"variables": what, "definitions": []}}
    if isinstance(what, str):           # a Python step: library function to call
        module = "warehouse" if what in ("set_cutoff", "check_warehouse_loaded", "clear_daily_flows") else "tracking"
        return {**common, "type": "custom_python", "params": {
            "script": f"from {module} import {what}\n\n{what}()\n", "envSelection": {"envMode": "INHERIT"},
            "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}
    kind = lambda i: ("MANAGED_FOLDER", folders[i].id) if i in folders else ("DATASET", i)
    builds = [{"type": t, "itemId": item, "partitionsSpec": ""} for t, item in map(kind, what)]
    return {**common, "type": "build_flowitem", "params": {
        "builds": builds, "jobType": mode, "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


START = [("Set the cut-off day (yesterday)", "set_cutoff"),
         ("Check the warehouse load is finished", "check_warehouse_loaded")]
NIGHTLY = [
    *START,
    ("Forecast with the production model and log it", ["inference_log"], "RECURSIVE_FORCED_BUILD"),
    ("Load the forecast into SQL", ["final_forecast"], "NON_RECURSIVE_FORCED_BUILD"),
]
SCENARIOS = {
    "Nightly batch": ("After each warehouse load: forecast the next 30, 60 and 90 days with the production model (the active "
                      "version of the saved model) into the folder forecast, log the run, then load it into the SQL "
                      "table final_forecast", "daily", NIGHTLY),
    "Retrain, test and promote": (
        "Train and test a new model with the forecaster the project variable forecaster names (tracked in Experiment "
        "Tracking, deployed as a saved model version), and activate it if it beats the production model. It does not "
        "forecast: the Nightly batch uses the active version", "monthly", [
            *START,
            ("Prepare the data (daily flows, clients)", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
            ("Train, test and deploy a new model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Promote if better", "apply_promotion"),
        ]),
    "Full rebuild of the daily flows": (
        "Run by hand: empties daily_flows and staging_log, then reads the whole history from the warehouse again "
        "(the nightly runs only read the new days). For example after a reversal cancels a posting of a day already "
        "staged. It does not forecast: run the Nightly batch after it", None, [
            *START,
            ("Empty the daily flows", "clear_daily_flows"),
            ("Read the whole history again", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
        ]),
}
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}
for name, (description, schedule, steps) in SCENARIOS.items():
    if name in existing_scenarios:     # run again: update its steps, keep its triggers and run history
        settings = project.get_scenario(existing_scenarios[name]).get_settings()
        settings.get_raw()["description"] = description
        before = {s["id"]: s for s in settings.get_raw()["params"].get("steps", [])}
        new_steps = [step(n, *s) for n, s in enumerate(steps)]
        for s in new_steps:              # numbers already set on the scenario's page are kept
            if s["type"] == "define_vars" and before.get(s["id"], {}).get("type") == "define_vars":
                kept = before[s["id"]]["params"]       # a new variable is added with its default,
                old = kept.get("variables", {})        # a variable no longer used is dropped
                kept["variables"] = {k: old.get(k, v) for k, v in s["params"]["variables"].items()}
                s["params"] = kept
        settings.get_raw()["params"]["steps"] = new_steps
        settings.save()
        print(f"Scenario updated: {name}")
        continue
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [step(n, *s) for n, s in enumerate(steps)]}})
    if schedule is None:                           # run by hand only
        print(f"Scenario ready: {name} (no trigger: run it by hand)")
        continue
    settings = scenario.get_settings()
    if schedule == "daily":
        settings.add_daily_trigger(hour=6, timezone="Asia/Riyadh")
    else:
        settings.add_monthly_trigger(day=5, hour=7, timezone="Asia/Riyadh")
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = "no trigger (not in this licence): add it on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. Results

To use another model (for example to roll back), activate its version in the saved model `cash_forecast` and run
the Nightly batch. To change when a model is promoted, change the project variables `promotion_*` (no code change).

In [ ]:
if staged_only:                                                 # only staging ran (HISTORY_MONTHS too short)
    clients_staged = dataiku.Dataset("clients").get_dataframe()
    print(f"Staged: {len(clients_staged):,} business clients, {cashflow.read_manifest()[1]['rows']:,} "
          "daily_flows rows (per client, day and category)")
    print("Staging log (one row per staging run: what was read from the warehouse):")
    display(dataiku.Dataset("staging_log").get_dataframe().sort_values("run_at"))
else:
    print("Production model (active version of the saved model cash_forecast):", tracking.production_model_id())
    print("Promotion log:")
    display(dataiku.Dataset("promotion_log").get_dataframe())
    print("Test on the last 6 periods of 30 days: TOTAL rows (net per client), category rows, and ALL together, by window (1 = days 1-30, 2 = 31-60, 3 = 61-90):")
    display(dataiku.Dataset("test_results").get_dataframe()[
        ["level", "window", "rows", "wape_model", "wape_baseline", "coverage_p10_p90", "below_p10", "above_p90"]])
    final = dataiku.Dataset("final_forecast").get_dataframe()
    print(f"Final forecast: one row per active client, {len(final)} rows, {final.shape[1]} columns. Status:",
          final["status"].value_counts().to_dict())
    one = final[final["status"] == "OK"].iloc[0]
    print(f"One client, {one['client_no']} (TOTAL and the first categories; empty: a category it does not have):")
    display(one.dropna().head(40).to_frame("value"))
    print("Inference log (one row per forecast run):")
    display(dataiku.Dataset("inference_log").get_dataframe().sort_values("run_at"))
    print("Staging log (one row per staging run: what was read from the warehouse):")
    display(dataiku.Dataset("staging_log").get_dataframe().sort_values("run_at"))